# Kang et al. (2023) 전차량 화재시험 재현 — 형상 · 센서 · 시각화

**대상:** S. Kang, M. Kwon, J.Y. Choi, S. Choi, *"Full-scale fire testing of battery
electric vehicles"*, **Applied Energy 332 (2023) 120497** (open access, CC BY)

`build_fds_input.ipynb`(KRISO 브리프 기반 예제)와 **별도 노트북**이다. 목적이 다르다.

| | `build_fds_input.ipynb` | 이 노트북 |
|---|---|---|
| 목적 | 1D↔3D 커플링 파이프라인 검증 | **실험 검증 (validation)** |
| 형상 | 브리프 6.2의 임의 치수 | **논문 Fig. 2·4의 실측 치수** |
| 좌표계 | FDS x = 차량 길이방향 | **논문 좌표계 그대로** |
| 센서 | 임의 배치 | **논문 Fig. 4(b)~(e)의 실제 센서 위치** |
| 정답 | 없음 | **논문 Table 4·6의 pHRR / THR / Δm** |

## 좌표계 — 논문 것을 그대로 쓴다

논문 Fig. 4(c)~(e)의 축 삼각형이 정의하는 계를 FDS에 그대로 옮겼다.
센서 좌표를 논문 숫자 그대로 쓸 수 있어 전사 오류가 생기지 않는다.

```
x : 차량 폭 방향   0 ~ 3,100 mm  (테스트 리그 폭). 차량 중심선 x = 1,550
y : 차량 길이 방향 0 ~ 5,200 mm  (리그 길이). y 증가 = 차량 전방
z : 연직          0 = 마운팅 플랫폼 상면.  천장 수냉 파이프 z = 2,270
```

> **주의:** 앞 노트북은 x가 차량 길이방향이었다. 여기서는 **y가 길이방향**이다.

## 산출물

| 파일 | 내용 |
|---|---|
| `<CHID>.fds` | FDS 입력 |
| `<CHID>_geometry.html` | **인터랙티브 3D** — 브라우저에서 회전·확대·호버, 커널 불필요 |
| `<CHID>_domain.png` | 정적 도메인 전체 단면 + 메쉬 경계 + 슬라이스면 |
| `params.json` | 이 입력을 만든 파라미터 전체 + QA 수치 + 논문 값 |

## 0. 경로 · 임포트

In [1]:
from __future__ import annotations

import json
import math
import subprocess
from dataclasses import dataclass, field, replace, asdict
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib

def _in_ipython_kernel():
    # 헤드리스(run_notebook.py)와 Jupyter 커널을 구분한다.
    try:
        from IPython import get_ipython
        ip = get_ipython()
        return ip is not None and ip.__class__.__name__ == "ZMQInteractiveShell"
    except Exception:
        return False

if _in_ipython_kernel():
    # 커널에서는 inline 백엔드라야 셀 출력에 그림이 뜬다.
    # Agg 로 고정하면 preview_domain() 이 Figure(...) 텍스트만 남긴다.
    get_ipython().run_line_magic("matplotlib", "inline")
else:
    matplotlib.use("Agg")           # 헤드리스: 디스플레이 없이 savefig 만
import matplotlib.pyplot as plt

ROOT    = Path("/scratch/x3319a05/FDS/electric_vehicle_battery")
INPUT   = ROOT / "input"
DATA_1D = INPUT / "data_1d"
REF     = INPUT / "ref_data"          # 논문 Fig. 8 디지타이즈 HRR 곡선
CASEDIR = INPUT / "cases"
RESULTS = ROOT / "results"
PAPER   = ROOT / "paper"

FDS_ENV = "/scratch/x3319a05/FDS/setup_docs/scripts/fds_env_kisti.sh"
FDS_BIN = "/scratch/x3319a05/FDS/Build/impi_intel_linux/fds_impi_intel_linux"

_trapz = getattr(np, "trapezoid", None) or np.trapz
mm = 1e-3   # 논문 치수는 mm, FDS는 m

try:
    import plotly.graph_objects as go
    HAVE_PLOTLY = True
except ImportError:
    HAVE_PLOTLY = False
    print("plotly 없음 -> 인터랙티브 3D 건너뜀.   pip install --user plotly")

print("논문:", [p.name for p in PAPER.glob("*.pdf")])

plotly 없음 -> 인터랙티브 3D 건너뜀.   pip install --user plotly
논문: ['Kang et al, 2023, Applied Energy, Full-scale fire testing of battery electric vehicles.pdf']


## 1. 논문에서 읽어낸 값

### 1.1 시험체 (Table 2, Table 3)

| 항목 | 값 | 비고 |
|---|---|---|
| 전장 × 전폭 × 전고 | **4,180 × 1,800 × 1,570 mm** | Test 2~4 BEV 공통 |
| 공차중량 | 1,685 kg (BEV_3) / 1,206 kg (차체만) / 449 kg (팩) | |
| LIB 용량 | 64.092 kWh (BEV_3), SOC 100 % | |
| 셀 | 파우치형 E63B (NCM622), 310 × 114 × 15 mm, 887.8 g, 60 Ah | |
| 모듈 | 10개. 하단 1·2·7·8·9·10 + 3·4, 상단(후방) 5·6 | Fig. 2(b) |
| 착화 | **Module_9 중앙 셀에 575 W 히팅시트** (90 × 65 mm) | Test 3·4 |

### 1.2 테스트 리그 (2.3절, Fig. 4)

한국 주차구획 최소규격(2.5 × 5.0 × 2.3 m)을 근거로 설계. 도면 실측은

| 항목 | 값 |
|---|---|
| 리그 폭 (x) | 3,100 mm (차량 중심선 x = 1,550) |
| 리그 길이 (y) | 5,200 mm |
| 천장 수냉 파이프 높이 | z = **2,270 mm** |
| 창문 높이 계측면 | z = **1,270 mm** |
| 후드 | 51.5 m² 원추형, 바닥 위 8.6 m. 배기덕트 3.8 m² |

### 1.3 센서 (Fig. 4(b)~(e))

**천장 열전대 6개 — z = 2,270 mm.** 차량 종방향 중앙면(x=1,550) 또는 좌측 도어창
라인(x=890) 위. y 좌표는 Fig. 4(c) 측면도의 치수 체인
`890 | 1040 | 160 | 640 | 560 | 800 | 1110` (합 5,200 = 리그 길이)에서 읽었다.

**창문 열전대 2개 — z = 1,270 mm.** 좌측 앞/뒤 도어창 중앙 높이.

**열유속계 4개 (Schmidt-Boelter SBG01) — z = 1,270 mm.** 리그 모서리에 정렬.
전방 보닛 / 트렁크 / 좌측 앞·뒷좌석 창을 향한다.

**로드셀 4개** — 시험체 질량 감소. FDS에서는 `&DUMP MASS_FILE=T` 로 대체.

> **불확실 항목:** 천장 열전대의 *이름↔y 대응*은 Fig. 4(b) 조감도의 라벨 순서와
> 부재 위치(보닛/윈드실드/루프/해치)로 추정했다. y 좌표 6개 자체는 측면도
> 치수라 확실하다. `HF_F`/`HF_R`의 x는 중앙면(1,550)으로 가정했다.

### 1.4 검증 정답 (Table 4, Table 6)

| Test | 시험체 | pHRR [MW] | THR [GJ] | m_ini [kg] | Δm [kg] | ΔH_c,eff [MJ/kg] |
|---|---|---|---|---|---|---|
| 1 | LIB 팩 (64 kWh) | 1.54 | 1.30 | 449 | 28 | 45.9 |
| 2 | 차체만 | 7.81 | 7.53 | 1,206 | 262 | 28.8 |
| 3 | BEV_2 (39 kWh) | 6.51 | 8.45 | 1,540 | 284 | 29.8 |
| 4 | **BEV_3 (64 kWh)** | **7.25** | **9.03** | 1,685 | 296 | 30.5 |

그 외: 연소 지속 **70분 이상**, pHRR 도달 **11~17분**, 화재성장계수
θ = 0.0085~0.020 MW/s² (`Q = θt²`).

**이 표가 `THICKNESS`와 `HEAT_OF_COMBUSTION`을 실측으로 고정해 준다** — 앞
노트북에서 플레이스홀더로 두고 튜닝하던 값들이다.

In [2]:
# ---- 논문 값 (전부 mm, MJ, kg 단위 그대로) -------------------------------
KANG = dict(
    vehicle=dict(length=4180., width=1800., height=1570.),            # Table 2
    rig=dict(width=3100., length=5200., z_pipe=2270., z_win=1270.),   # Fig. 4
    centreline_x=1550.,
    window_line_x=890.,                                               # Fig. 4(d) 890|660
    tc_chain=[890., 1040., 160., 640., 560., 800., 1110.],            # Fig. 4(c)
    targets={                                                         # Table 4 / Table 6
        "LIB pack": dict(pHRR_MW=1.54, THR_GJ=1.30, m_ini=449.,  dm=28.,  dHc=45.9),
        "BEV body": dict(pHRR_MW=7.81, THR_GJ=7.53, m_ini=1206., dm=262., dHc=28.8),
        "BEV_2":    dict(pHRR_MW=6.51, THR_GJ=8.45, m_ini=1540., dm=284., dHc=29.8),
        "BEV_3":    dict(pHRR_MW=7.25, THR_GJ=9.03, m_ini=1685., dm=296., dHc=30.5),
    },
    t_peak_min=(11., 17.), burn_min=70., theta_MW_s2=(0.0085, 0.020),
)

_y = np.cumsum(KANG["tc_chain"][:-1])          # 측면도 치수 체인 -> 천장 TC 의 y [mm]
assert abs(sum(KANG["tc_chain"]) - KANG["rig"]["length"]) < 1e-6, "치수 체인 합 != 리그 길이"
CL, WL = KANG["centreline_x"], KANG["window_line_x"]
RIG_L, RIG_W = KANG["rig"]["length"], KANG["rig"]["width"]
Z_PIPE, Z_WIN = KANG["rig"]["z_pipe"], KANG["rig"]["z_win"]

# 이름 배정은 Fig. 4(b) 조감도 라벨 순서 + 부재 위치 기준 (1.3절 '불확실 항목')
#   전방(y 큼) -> 후방 : 보닛, 윈드실드, 앞도어창, 루프, 뒷도어창, 해치유리
SENSORS_TC_CEIL = [                     # (ID, x, y)  z = 2270
    ("TC_GB",   CL, _y[5]),             # 4090  보닛 위
    ("TC_GW",   CL, _y[4]),             # 3290  윈드실드 위
    ("TC_GFW",  WL, _y[3]),             # 2730  앞좌석 창 위
    ("TC_GR",   CL, _y[2]),             # 2090  루프 위
    ("TC_GRW",  WL, _y[1]),             # 1930  뒷좌석 창 위
    ("TC_GCRW", CL, _y[0]),             #  890  해치(리어윈도) 위
]
SENSORS_TC_WIN = [("TC_FW", WL, _y[3]), ("TC_RW", WL, _y[1])]     # z = 1270
SENSORS_HF = [                          # (ID, x, y, 바라보는 방향)  z = 1270
    ("HF_F",  CL,  RIG_L, (0, -1, 0)),  # 전방 모서리 -> 보닛
    ("HF_R",  CL,  0.0,   (0, +1, 0)),  # 후방 모서리 -> 트렁크
    ("HF_FD", 0.0, _y[3], (+1, 0, 0)),  # 좌측 모서리 -> 앞좌석 창
    ("HF_RD", 0.0, _y[1], (+1, 0, 0)),  # 좌측 모서리 -> 뒷좌석 창
]

# ---- 논문 좌표계 -> FDS 좌표계 -------------------------------------------
# 차량 중심(= 리그 중심)이 도메인 원점 (0,0) 에 오도록 평행이동한다.
# 논문 값은 전부 mm 로 그대로 두고, FDS 로 나갈 때만 이 두 함수를 통과시킨다.
PAPER_ORIGIN = (CL * mm, RIG_L / 2 * mm)          # (1.550, 2.600) m
px = lambda x_mm: round(x_mm * mm - PAPER_ORIGIN[0], 6)
py = lambda y_mm: round(y_mm * mm - PAPER_ORIGIN[1], 6)

print(f"논문 원점 -> FDS 원점 평행이동: ({-PAPER_ORIGIN[0]:+.3f}, {-PAPER_ORIGIN[1]:+.3f}) m")
print("천장 열전대 y [mm]:", list(_y.astype(int)))
print(f"  {'ID':<8} {'논문 x':>8} {'논문 y':>8} | {'FDS x':>7} {'FDS y':>7} {'z':>7}")
for n, x, y in SENSORS_TC_CEIL:
    print(f"  {n:<8} {x:8.0f} {y:8.0f} | {px(x):7.3f} {py(y):7.3f} {Z_PIPE*mm:7.3f}")
for n, x, y in SENSORS_TC_WIN:
    print(f"  {n:<8} {x:8.0f} {y:8.0f} | {px(x):7.3f} {py(y):7.3f} {Z_WIN*mm:7.3f}")
for n, x, y, o in SENSORS_HF:
    print(f"  {n:<8} {x:8.0f} {y:8.0f} | {px(x):7.3f} {py(y):7.3f} {Z_WIN*mm:7.3f}  향={o}")

천장 열전대 y [mm]: [np.int64(890), np.int64(1930), np.int64(2090), np.int64(2730), np.int64(3290), np.int64(4090)]
  TC_GB    x=   1550  y=   4090  z=2270
  TC_GW    x=   1550  y=   3290  z=2270
  TC_GFW   x=    890  y=   2730  z=2270
  TC_GR    x=   1550  y=   2090  z=2270
  TC_GRW   x=    890  y=   1930  z=2270
  TC_GCRW  x=   1550  y=    890  z=2270
  TC_FW    x=    890  y=   2730  z=1270
  TC_RW    x=    890  y=   1930  z=1270
  HF_F     x=   1550  y=   5200  z=1270  향=(0, -1, 0)
  HF_R     x=   1550  y=      0  z=1270  향=(0, 1, 0)
  HF_FD    x=      0  y=   2730  z=1270  향=(1, 0, 0)
  HF_RD    x=      0  y=   1930  z=1270  향=(1, 0, 0)


## 2. 케이스 파라미터

차체 물성 중 **`HEAT_OF_COMBUSTION`과 연료 총량은 논문 Table 6에서 고정**한다.
나머지(밀도·비열·전도도·기화열·비등점)는 추정값이며, 이 중 **기화열이 연소 속도를
지배**하는 주 튜닝 노브다 (pHRR 7.81 MW, 도달 11~17분에 맞춤).

In [3]:
@dataclass
class Grid:
    # 차량 중심이 도메인 원점 (0,0). 좌우 대칭 스냅이 성립하도록 원점 대칭 도메인.
    xmin: float = -4.00; xmax: float = 4.00     # 차량 중심 +-4.0 m
    ymin: float = -5.00; ymax: float = 5.00     # 차량 중심 +-5.0 m
    zmin: float =  0.00; zmax: float = 6.00
    dx: float = 0.125
    nmesh: tuple = (2, 2, 1)
    cuts: tuple = (None, None, None)            # 축별 명시적 분할면 [m]

    def __post_init__(self):
        # x=0, y=0 이 격자선 위에 있어야 좌/우 미러가 정확히 같은 셀 수로 떨어진다.
        for nm, v in (("xmin", self.xmin), ("ymin", self.ymin)):
            if abs(v / self.dx - round(v / self.dx)) > 1e-9:
                raise ValueError(f"{nm}={v} 가 dx={self.dx} 의 정수배가 아니다 "
                                 f"-> 원점이 격자선에서 벗어나 좌우 대칭이 깨진다")

    @property
    def ijk(self):
        return tuple(round((b - a) / self.dx) for a, b in
                     ((self.xmin, self.xmax), (self.ymin, self.ymax), (self.zmin, self.zmax)))

    @property
    def ncell(self):
        i, j, k = self.ijk
        return i * j * k

    @property
    def nranks(self):
        return self.nmesh[0] * self.nmesh[1] * self.nmesh[2]


# =========================================================================
#  차량은 타이어 / 유리 / 프레임 / 배터리팩 네 부품으로 나뉘고,
#  각 부품이 자기 형상 파라미터와 자기 재료 파라미터를 독립으로 갖는다.
# =========================================================================

@dataclass
class Vehicle:
    # 전체 치수와 수직 레벨만 담당한다. 부재별 파라미터는 아래 dataclass 로.
    # x = 폭, y = 길이(후->전), z = 높이. 차량 중심 = (0, 0), 지면 = z 0.
    length: float = 4.180        # 전장 (y)
    width:  float = 1.800        # 전폭 (x)
    height: float = 1.570        # 전고 (z) = 루프 상면

    z_floor: float = 0.450       # 차체 바닥 하면. 팩과 띄우려고 올렸다.
    z_belt:  float = 1.000       # 벨트라인 = 유리 하단

    # 캐빈(사람이 타는 구역) 배치
    cabin_len:   float = 2.000   # 캐빈 길이 (y)
    cabin_dy:    float = 0.000   # 차량 중심 대비 오프셋 (0 = 전후 대칭)
    cabin_inset: float = 0.150   # 편측 폭 축소 -> 캐빈 반폭 = width/2 - inset
    pillar_b:    float = 0.250   # B필러 폭 (y). y 대칭을 위해 2셀(=2*dx) 권장


@dataclass
class Tire:
    # 235/45 R18. 외경 = 림지름 + 2 x 편평비 x 단면폭
    section_width_mm: float = 235.0
    aspect_ratio_pct: float = 45.0
    rim_inch:         float = 18.0
    wheelbase:        float = 2.600     # 축거 -> 축 위치 y = +-wheelbase/2
    outer_x:          float = 0.850     # 타이어 바깥면 |x| (차체 반폭 0.9 안쪽)
    mass_kg:          float = 40.0      # 4짝 합. 타이어 THICKNESS 역산에 쓴다.
    # 재료 (프레임과 독립. 화학종은 CAR_BODY_FUEL 로 공유해 REAC 를 늘리지 않는다)
    density:                float = 1100.0
    specific_heat:          float = 1.80
    conductivity:           float = 0.25
    boiling_temperature:    float = 420.0
    heat_of_reaction:       float = 2200.0
    absorption_coefficient: float = 1000.0
    emissivity:             float = 0.95

    @property
    def width(self):
        return self.section_width_mm * mm

    @property
    def diameter(self):
        return (self.rim_inch * 0.0254
                + 2.0 * self.section_width_mm * self.aspect_ratio_pct / 100.0 * mm)


@dataclass
class Glass:
    # 유리는 형상에서 빼지 않는다. 프레임과 다른 SURF 로 '채워' 넣어
    # 여기 파라미터만 바꾸면 물성을 독립으로 손볼 수 있게 한다.
    thickness:     float = 0.006
    density:       float = 2500.0
    specific_heat: float = 0.84
    conductivity:  float = 1.05
    emissivity:    float = 0.90
    combustible:   bool  = False     # True 면 프레임 재료로 연소시킨다
    # 논문 2.4절: 좌측(=x 음수) 앞·뒷 도어창을 반쯤 열어 캐빈을 환기시켰다.
    # 창을 '내린' 만큼 유리를 채우지 않는다. 창문 열전대 z=1.27 이 여기 놓인다.
    open_windows: tuple  = ("FL", "RL")
    open_height:  float  = 0.250     # 유리 하단에서 이만큼 내림. 0 이면 전부 닫힘


@dataclass
class Frame:
    # 차체 프레임 (금속 + 내장 가연물의 등가 재료)
    shell: float = 0.125             # 쉘 두께 [m] (형상). 내부는 비어 있다.
    density:                float = 900.0
    specific_heat:          float = 1.50
    conductivity:           float = 0.20
    boiling_temperature:    float = 380.0
    heat_of_reaction:       float = 1800.0   # <- pHRR / 성장속도 주 튜닝 노브
    absorption_coefficient: float = 1000.0
    emissivity:             float = 0.90
    formula:            str   = "C6H10O1"
    heat_of_combustion: float = 28800.0      # Table 6 (BEV body) 실측
    soot_yield:         float = 0.10
    co_yield:           float = 0.05
    radiative_fraction: float = 0.35
    fuel_mass_kg:       float = 262.0        # Table 6 소실질량 (타이어 포함)


@dataclass
class Pack:
    # 배터리 팩 - 셀/모듈 무시하고 단순 직육면체.
    # 6면 전부에 벤트가스 경계조건을 주므로, 지면과 차체 바닥 어느 쪽에도
    # 닿지 않게 띄워 놓는다 (닿은 면은 기체와 접하지 않아 &VENT 가 죽는다).
    length: float = 1.600        # y
    width:  float = 1.200        # x
    height: float = 0.150        # z
    # 위치는 '차체 바닥 하면 - 팩 상면' 간격으로 준다. 바닥을 올리거나 내려도
    # 팩이 따라 움직이므로 둘이 어긋날 일이 없다.
    gap_to_floor: float = 0.100  # 팩 상면 ~ 차체 바닥 하면 [m]. 최소 1셀 권장.
    z_bot: Optional[float] = None  # 직접 지정하고 싶을 때만. None -> 간격에서 역산
    dy:    float = 0.000         # 차량 중심 대비 오프셋 (0 = 대칭)


@dataclass
class VentLayout:
    # 논문의 열폭주는 중앙 Module_9 에서 시작해 바깥으로 전파한다.
    zone_len: float = 0.50
    sides: tuple = ("XMIN", "XMAX")   # 팩 좌·우 측면에서 방출


@dataclass
class PackHRR:
    """팩 발열률을 ref_data 의 실측 곡선으로 직접 지정한다.

    csv 는 [시간(min), HRR(MW)] 2열. 팩 6면에 같은 HRRPUA 를 주면 각 면 기여가
    면적에 비례하고 6면 합이 곡선값과 일치한다 (합 = HRRPUA x 총면적).
    """
    csv:     str   = "pack_HRR.csv"
    t_unit:  float = 60.0        # csv 시간 -> 초
    q_unit:  float = 1000.0      # csv HRR  -> kW
    scale:   float = 1.0         # 곡선 전체 배율 (튜닝 노브)
    clip_negative: bool = True   # 디지타이즈 잡음으로 생긴 음수 HRR 제거


@dataclass
class BatteryGas:
    composition: dict = field(default_factory=lambda: {
        "H2": 28.0, "CO": 23.0, "CO2": 28.0, "CH4": 12.0, "C2H4": 9.0})
    soot_yield: float = 0.02
    co_yield: float = 0.05
    radiative_fraction: float = 0.15
    heat_of_combustion: Optional[float] = None   # None -> 조성에서 자동 유도
    track_hf: bool = True


@dataclass
class Case:
    chid: str
    title: str
    t_end: float = 1800.0
    tmpa: float = 20.0
    grid:  Grid  = field(default_factory=Grid)
    veh:   Vehicle = field(default_factory=Vehicle)
    tire:  Tire  = field(default_factory=Tire)
    glass: Glass = field(default_factory=Glass)
    frame: Frame = field(default_factory=Frame)
    pack:  Pack  = field(default_factory=Pack)
    vent:  VentLayout = field(default_factory=VentLayout)
    gas:   BatteryGas = field(default_factory=BatteryGas)
    hrr:   PackHRR    = field(default_factory=PackHRR)
    ramp_eps: float = 0.004
    dt_hrr: float = 1.0; dt_devc: float = 1.0
    dt_slcf: float = 5.0; dt_bndf: float = 20.0
    walltime: str = "24:00:00"


## 3. 지오메트리 빌더

FDS는 `OBST`/`VENT` 좌표를 격자에 스냅한다. **스냅 후 좌표**가 실제로 계산에
쓰이는 값이므로, 시각화도 QA도 전부 스냅 후 값으로 한다.

캐빈은 **속이 빈 쉘**이다. 논문 3.1절이 "pHRR/THR을 지배하는 것은 캐빈 내부
가연물"이라고 명시하는데, 통짜 solid로 두면 내부 표면이 노출면적에서 빠지고
환기(반쯤 열린 좌측 도어창)도 모사할 수 없다.

In [4]:
def snap_box(xb, g: Grid, min_cells=1):
    # 원점(0) 기준으로 스냅한다. Grid.__post_init__ 이 원점을 격자선 위에
    # 강제하므로, round() 가 홀수 0.5 에서도 부호대칭이라 좌/우 미러가
    # 정확히 같은 셀 수로 떨어진다 (round(2.5)=2, round(-2.5)=-2).
    out = []
    for a in range(3):
        lo, hi = xb[2 * a], xb[2 * a + 1]
        i0 = round(lo / g.dx); i1 = round(hi / g.dx)
        if abs(hi - lo) < 1e-12:  i1 = i0            # 평면 유지
        elif i1 - i0 < min_cells: i1 = i0 + min_cells
        out += [i0 * g.dx, i1 * g.dx]
    return tuple(round(v, 6) for v in out)


def snap_scalar(v, g: Grid, axis=0):
    return round(round(v / g.dx) * g.dx, 6)


def mesh_lines(g: Grid):
    ijk = g.ijk; lo = (g.xmin, g.ymin, g.zmin)
    def split(n, nm, o, ex, ax):
        if ex:
            bnd = sorted({0, n} | {round((snap_scalar(v, g, ax) - o) / g.dx) for v in ex})
        else:
            base, rem = divmod(n, nm); bnd, acc = [0], 0
            for m in range(nm):
                acc += base + (1 if m < rem else 0); bnd.append(acc)
        return [(bnd[i+1]-bnd[i], o+bnd[i]*g.dx, o+bnd[i+1]*g.dx) for i in range(len(bnd)-1)]
    cx = split(ijk[0], g.nmesh[0], lo[0], g.cuts[0], 0)
    cy = split(ijk[1], g.nmesh[1], lo[1], g.cuts[1], 1)
    cz = split(ijk[2], g.nmesh[2], lo[2], g.cuts[2], 2)
    lines, n = [], 0
    for a in cx:
        for b in cy:
            for c_ in cz:
                n += 1
                lines.append(f"&MESH ID='M{n:02d}', IJK={a[0]},{b[0]},{c_[0]}, "
                             f"XB={a[1]:.4f},{a[2]:.4f}, {b[1]:.4f},{b[2]:.4f}, "
                             f"{c_[1]:.4f},{c_[2]:.4f} /")
    return lines, dict(x=[a[2] for a in cx[:-1]], y=[b[2] for b in cy[:-1]],
                       z=[c[2] for c in cz[:-1]])


SURF_FRAME, SURF_GLASS, SURF_TIRE, SURF_PACK = "CAR_BODY", "CAR_GLASS", "TIRE", "PACK_CASE"


def pack_z(c: Case):
    """팩의 (하면, 상면) z. 형상 빌더와 슬라이스면이 이 하나를 공유한다."""
    v, pk = c.veh, c.pack
    z_top = (pk.z_bot + pk.height) if pk.z_bot is not None else (v.z_floor - pk.gap_to_floor)
    return z_top - pk.height, z_top


def build_geometry(c: Case):
    """차량을 타이어 -> 하부 차체(속 빈 쉘) -> 캐빈(프레임+유리) 순으로 쌓는다.

    - 모든 부재는 x=0 (차량 중심선)에 대해 좌우 대칭이다.
    - 하부 차체와 캐빈은 두께 `frame.shell` 의 판으로만 이루어져 내부가 비어 있다.
    - 캐빈 프레임(루프 + 벨트레일 링 + 6개 필러)은 서로 맞닿아 하나로 연결된다.
    - 유리는 개구로 비우지 않고 SURF_GLASS 로 채운다 (열린 창만 예외).
    """
    g, v, fr, gl, ti, pk = c.grid, c.veh, c.frame, c.glass, c.tire, c.pack
    t   = fr.shell
    hw  = v.width / 2                         # 차체 반폭
    hl  = v.length / 2                        # 차체 반길이
    chw = hw - v.cabin_inset                  # 캐빈 반폭
    cy0 = v.cabin_dy - v.cabin_len / 2        # 캐빈 후단
    cy1 = v.cabin_dy + v.cabin_len / 2        # 캐빈 전단
    zf, zb, zr = v.z_floor, v.z_belt, v.height
    hb  = v.pillar_b / 2                      # B필러 반폭

    obst = []
    add = lambda n, xb, s, col: obst.append((n, xb, s, col))

    # ---- 1. 타이어 (하부에 먼저) -----------------------------------------
    # 먼저 스냅해서 휠아치 구간을 확정한다. 그래야 차체 판을 아치에 맞춰
    # 잘라낼 수 있고, 타이어와 차체가 셀을 겹쳐 먹지 않는다.
    td, tw, ya = ti.diameter, ti.width, ti.wheelbase / 2
    for tag, yc_ in (("R", -ya), ("F", ya)):
        for sd, xa, xb_ in (("L", -ti.outer_x, -ti.outer_x + tw),
                            ("R",  ti.outer_x - tw,  ti.outer_x)):
            add(f"TIRE_{tag}{sd}",
                snap_box((xa, xb_, yc_ - td/2, yc_ + td/2, 0.0, td), g),
                SURF_TIRE, "BLACK")
    arches = sorted({(b[2], b[3]) for _, b, _, _ in obst})      # 휠아치 y 구간 2개
    z_arch = max(b[5] for _, b, _, _ in obst)                   # 타이어 상단
    x_in   = min(min(abs(b[0]), abs(b[1])) for _, b, _, _ in obst)   # 타이어 안쪽면 |x|

    def _segments(lo, hi, cuts):
        # [lo,hi] 에서 cuts 구간을 뺀 나머지 구간들
        segs, cur = [], lo
        for a, b_ in sorted(cuts):
            a, b_ = max(a, lo), min(b_, hi)
            if b_ <= cur: continue
            if a > cur: segs.append((cur, a))
            cur = max(cur, b_)
        if cur < hi: segs.append((cur, hi))
        return segs

    # ---- 2. 하부 차체: 단순 직육면체를 두께 t 의 쉘로 (내부는 빈 공간) ----
    # 휠아치에서는 바닥판을 타이어 안쪽까지만 깔고, 측면판은 타이어 위에서
    # 시작한다. 아치 구멍은 타이어가 막으므로 실내는 그대로 밀폐된다.
    for i_, (ya_, yb_) in enumerate(_segments(-hl, hl, arches), 1):
        add(f"FLOOR_{i_}",  (-hw, hw, ya_, yb_, zf, zf + t),   SURF_FRAME, "GRAY 60")
        add(f"SIDE_L_{i_}", (-hw, -hw + t, ya_, yb_, zf, zb),  SURF_FRAME, "SILVER")
        add(f"SIDE_R_{i_}", ( hw - t, hw,  ya_, yb_, zf, zb),  SURF_FRAME, "SILVER")
    for i_, (ya_, yb_) in enumerate(arches, 1):
        add(f"FLOOR_ARCH_{i_}",  (-x_in, x_in, ya_, yb_, zf, zf + t), SURF_FRAME, "GRAY 60")
        add(f"SIDE_L_ARCH_{i_}", (-hw, -hw + t, ya_, yb_, z_arch, zb), SURF_FRAME, "SILVER")
        add(f"SIDE_R_ARCH_{i_}", ( hw - t, hw,  ya_, yb_, z_arch, zb), SURF_FRAME, "SILVER")
    add("END_REAR",  (-hw, hw, -hl, -hl + t, zf, zb),         SURF_FRAME, "SILVER")
    add("END_FRONT", (-hw, hw,  hl - t, hl, zf, zb),          SURF_FRAME, "SILVER")
    # 벨트라인 데크. 캐빈 앞(보닛)·뒤(트렁크)·좌우(사이드실)를 덮는다.
    # 네 판 모두 캐빈 레일 폭(t)만큼 안쪽까지 물려야 레일이 데크 '바로 위' 셀에
    # 얹힌다. 모서리로만 닿으면 6-이웃 기준으로 캐빈이 하부 차체에서 떨어진다.
    # 캐빈 안쪽은 여전히 뚫려 있어 실내와 하부가 하나의 빈 공간으로 이어진다.
    add("HOOD",   (-hw, hw, cy1 - t, hl,  zb - t, zb),        SURF_FRAME, "SILVER")
    add("TRUNK",  (-hw, hw, -hl, cy0 + t, zb - t, zb),        SURF_FRAME, "SILVER")
    # 사이드실은 벨트레일 바로 아래까지(-chw+t) 덮는다. 여기서 끊기면 캐빈
    # 프레임이 하부 차체와 모서리로만 닿아 6-이웃 기준으로 분리된다.
    add("SILL_L", (-hw, -chw + t, cy0, cy1, zb - t, zb),      SURF_FRAME, "SILVER")
    add("SILL_R", ( chw - t,  hw, cy0, cy1, zb - t, zb),      SURF_FRAME, "SILVER")

    # ---- 3. 캐빈 프레임 - 전부 맞닿아 하나로 연결된다 --------------------
    #   루프판(위) + 벨트레일 링(아래) + 6개 필러(수직) 가 서로 접한다.
    add("ROOF",       (-chw, chw, cy0, cy1, zr - t, zr),      SURF_FRAME, "SILVER")
    add("RAIL_L",     (-chw, -chw + t, cy0, cy1, zb, zb + t), SURF_FRAME, "SILVER")
    add("RAIL_R",     ( chw - t, chw, cy0, cy1, zb, zb + t),  SURF_FRAME, "SILVER")
    add("RAIL_REAR",  (-chw, chw, cy0, cy0 + t, zb, zb + t),  SURF_FRAME, "SILVER")
    add("RAIL_FRONT", (-chw, chw, cy1 - t, cy1, zb, zb + t),  SURF_FRAME, "SILVER")
    for sd, xa, xb_ in (("L", -chw, -chw + t), ("R", chw - t, chw)):
        add(f"APILLAR_{sd}", (xa, xb_, cy1 - t, cy1, zb, zr), SURF_FRAME, "SILVER")
        add(f"BPILLAR_{sd}", (xa, xb_, v.cabin_dy - hb, v.cabin_dy + hb, zb, zr),
            SURF_FRAME, "SILVER")
        add(f"CPILLAR_{sd}", (xa, xb_, cy0, cy0 + t, zb, zr), SURF_FRAME, "SILVER")

    # ---- 4. 유리 - 개구로 비우지 않고 별도 SURF 로 채운다 ----------------
    zg0, zg1 = zb + t, zr - t                 # 유리 하단 / 상단
    glass_boxes = [
        ("GLASS_WINDSHIELD", (-chw + t, chw - t, cy1 - t, cy1, zg0, zg1)),
        ("GLASS_BACKLIGHT",  (-chw + t, chw - t, cy0, cy0 + t, zg0, zg1)),
    ]
    for sd, xa, xb_ in (("L", -chw, -chw + t), ("R", chw - t, chw)):
        for tag, ya_, yb_ in (("F", v.cabin_dy + hb, cy1 - t),
                              ("R", cy0 + t, v.cabin_dy - hb)):
            key = f"{tag}{sd}"
            z0 = zg0 + (gl.open_height if key in gl.open_windows else 0.0)
            if z0 >= zg1 - 1e-9:              # 창을 끝까지 내린 경우
                continue
            glass_boxes.append((f"GLASS_{key}", (xa, xb_, ya_, yb_, z0, zg1)))
    gsurf = SURF_FRAME if gl.combustible else SURF_GLASS
    for n, xb in glass_boxes:
        add(n, xb, gsurf, "SKY BLUE")

    # ---- 5. 배터리 팩 - 단순 직육면체 ------------------------------------
    pz0, pz1 = pack_z(c)
    pack = ("PACK", (-pk.width / 2, pk.width / 2,
                     pk.dy - pk.length / 2, pk.dy + pk.length / 2,
                     pz0, pz1), SURF_PACK, "GRAY 30")

    obst = [(n, snap_box(xb, g), s, col) for n, xb, s, col in obst]   # 타이어는 이미 스냅됨(멱등)
    pack = (pack[0], snap_box(pack[1], g), pack[2], pack[3])

    # ---- 6. 벤트: 팩 6면 전체에 같은 경계조건 ---------------------------
    # 면을 존으로 세분화하지 않는다. 팩을 차체 하부에서 띄워 두었으므로
    # 6면이 모두 기체와 접하고, 각 면에 &VENT 를 그대로 붙일 수 있다.
    P = pack[1]
    faces = [("XMIN", (P[0], P[0], P[2], P[3], P[4], P[5])),
             ("XMAX", (P[1], P[1], P[2], P[3], P[4], P[5])),
             ("YMIN", (P[0], P[1], P[2], P[2], P[4], P[5])),
             ("YMAX", (P[0], P[1], P[3], P[3], P[4], P[5])),
             ("ZMIN", (P[0], P[1], P[2], P[3], P[4], P[4])),
             ("ZMAX", (P[0], P[1], P[2], P[3], P[5], P[5]))]

    def _patch_area(q):
        e = [v for v in (abs(q[1]-q[0]), abs(q[3]-q[2]), abs(q[5]-q[4])) if v > 1e-12]
        return e[0] * e[1] if len(e) == 2 else 0.0

    vents = [dict(name="PACK", patches=[q for _, q in faces],
                  faces=[n for n, _ in faces],
                  area=sum(_patch_area(q) for _, q in faces))]

    return dict(obst=obst, pack=pack, vents=vents,
                cabin=snap_box((-chw, chw, cy0, cy1, zb, zr), g),
                bbox=dict(x=(-hw, hw), y=(-hl, hl), z=(0.0, zr)))


fmt_xb = lambda xb: ",".join(f"{v:.4f}" for v in xb)


def symmetry_check(geo, surfs=None):
    """x=0 (차량 중심선)에 대한 좌우 대칭 검사. surfs 를 주면 그 SURF 만 본다."""
    items = [(n, xb, s) for n, xb, s, _ in geo["obst"]]
    items.append((geo["pack"][0], geo["pack"][1], geo["pack"][2]))
    if surfs is not None:
        items = [it for it in items if it[2] in surfs]
    key = lambda xb: tuple(round(q, 6) for q in xb)
    have = {key(xb) for _, xb, _ in items}
    return [n for n, xb, _ in items if key((-xb[1], -xb[0]) + tuple(xb[2:])) not in have]


## 4. 노출면적 실측 → `THICKNESS` 확정

논문 Table 6이 **차체 소실질량 262 kg**을 준다. 격자에 래스터화해서 **가스와 접한
면만** 세어 실제 노출 연소면적을 구하고 그것으로 나누면 `THICKNESS`가 추정이 아니라
**역산값**이 된다.

```
262 kg = ρ × (A_body × t_body + A_tire × t_tire),   t_tire = 1.7 × t_body
```

In [5]:
def rasterize(boxes, g: Grid):
    nx, ny, nz = g.ijk
    m = np.zeros((nx, ny, nz), bool)
    for xb in boxes:
        i0 = max(0, round((xb[0]-g.xmin)/g.dx)); i1 = min(nx, round((xb[1]-g.xmin)/g.dx))
        j0 = max(0, round((xb[2]-g.ymin)/g.dx)); j1 = min(ny, round((xb[3]-g.ymin)/g.dx))
        k0 = max(0, round((xb[4]-g.zmin)/g.dx)); k1 = min(nz, round((xb[5]-g.zmin)/g.dx))
        m[i0:i1, j0:j1, k0:k1] = True
    return m


def exposed_area(mask, solid, g: Grid, ground=True):
    p = np.zeros(np.array(solid.shape) + 2, bool)
    p[1:-1, 1:-1, 1:-1] = solid
    if ground: p[:, :, 0] = True
    tot = 0
    for ax, off in ((0,-1),(0,1),(1,-1),(1,1),(2,-1),(2,1)):
        sl = [slice(1,-1)]*3
        sl[ax] = slice(1+off, p.shape[ax]-1+off)
        tot += int(np.count_nonzero(mask & ~p[tuple(sl)]))
    return tot * g.dx**2


def _grow(seed, free, cap=4000):
    # free 안에서 seed 를 6-이웃으로 최대한 확장한다 (scipy 없이).
    cur = seed & free
    for _ in range(cap):
        n = cur.sum()
        d = cur.copy()
        d[1:, :, :] |= cur[:-1, :, :]; d[:-1, :, :] |= cur[1:, :, :]
        d[:, 1:, :] |= cur[:, :-1, :]; d[:, :-1, :] |= cur[:, 1:, :]
        d[:, :, 1:] |= cur[:, :, :-1]; d[:, :, :-1] |= cur[:, :, 1:]
        cur = d & free
        if cur.sum() == n:
            break
    return cur


def n_components(mask):
    # 6-이웃 연결 성분 개수. 프레임이 하나로 붙어 있는지 확인하는 용도.
    rem, n = mask.copy(), 0
    while rem.any():
        n += 1
        seed = np.zeros_like(mask)
        seed[tuple(np.argwhere(rem)[0])] = True
        rem &= ~_grow(seed, mask)
    return n


def enclosed_void(solid, g: Grid):
    # 도메인 바깥에서 기체를 타고 들어갈 수 없는 셀 = 밀폐된 실내 공간.
    free = ~solid
    seed = np.zeros_like(free)
    seed[0, :, :] = seed[-1, :, :] = True
    seed[:, 0, :] = seed[:, -1, :] = True
    seed[:, :, -1] = True                       # 바닥(z=0)은 지면이라 제외
    outside = _grow(seed & free, free)
    return free & ~outside


def _snap_report(c: Case, geo):
    # dx 로 안 떨어지는 치수는 격자에 스냅되며 값이 바뀐다. 조용히 넘어가면
    # 나중에 "왜 팩이 요청한 치수와 다르지" 로 돌아오므로 여기서 드러낸다.
    P_ = geo["pack"][1]
    box = {n: xb for n, xb, _, _ in geo["obst"]}
    tl = box["TIRE_FL"]
    rows = [("차량 전폭",   c.veh.width,     max(b[1] for b in box.values())
                                          - min(b[0] for b in box.values())),
            ("차량 전장",   c.veh.length,    max(b[3] for b in box.values())
                                          - min(b[2] for b in box.values())),
            ("팩 폭",       c.pack.width,    P_[1] - P_[0]),
            ("팩 길이",     c.pack.length,   P_[3] - P_[2]),
            ("팩 높이",     c.pack.height,   P_[5] - P_[4]),
            ("팩-바닥 간격", c.pack.gap_to_floor,
                             min(b[4] for n, b in box.items() if n.startswith("FLOOR")) - P_[5]),
            ("타이어 폭",   c.tire.width,    tl[1] - tl[0]),
            ("타이어 외경", c.tire.diameter, tl[5] - tl[4])]
    return [(l, w, gt) for l, w, gt in rows if abs(gt - w) > 1e-9]


def geometry_qa(c: Case, geo=None, verbose=True):
    geo = geo or build_geometry(c)
    g, fr, ti, gl = c.grid, c.frame, c.tire, c.glass
    pick = lambda s: [xb for _, xb, s_, _ in geo["obst"] if s_ == s]
    m_frame = rasterize(pick(SURF_FRAME), g)
    m_glass = rasterize(pick(SURF_GLASS), g) & ~m_frame
    m_tire  = rasterize(pick(SURF_TIRE),  g) & ~(m_frame | m_glass)
    m_pack  = rasterize([geo["pack"][1]], g) & ~(m_frame | m_glass | m_tire)
    solid = m_frame | m_glass | m_tire | m_pack
    a_frame = exposed_area(m_frame, solid, g)
    a_glass = exposed_area(m_glass, solid, g)
    a_tire  = exposed_area(m_tire,  solid, g)
    a_pack  = exposed_area(m_pack,  solid, g)

    # 타이어와 프레임을 각자 질량으로 독립 역산한다 (부품별 파라미터 분리).
    m_tire_kg  = ti.mass_kg
    m_frame_kg = fr.fuel_mass_kg - m_tire_kg
    t_tire  = m_tire_kg  / (ti.density * a_tire)  if a_tire  > 0 else 0.0
    t_frame = m_frame_kg / (fr.density * a_frame) if a_frame > 0 else 0.0
    thr_body = fr.fuel_mass_kg * fr.heat_of_combustion / 1e6

    q = KANG["targets"]["BEV_3"]["pHRR_MW"] * 1000.
    dstar = (q / (1.204*1.005*293.*math.sqrt(9.81)))**0.4
    asym      = symmetry_check(geo, surfs={SURF_FRAME, SURF_TIRE, SURF_PACK})
    asym_gl   = symmetry_check(geo, surfs={SURF_GLASS})
    n_frame_c = n_components(m_frame)
    # 팩 6면이 정말 기체와 접하는지 (=&VENT 가 살아있는지) 확인한다.
    gas_ = ~solid
    pf = []
    for lab, ax, off in (("ZMIN", 2, -1), ("ZMAX", 2, 1), ("XMIN", 0, -1),
                         ("XMAX", 0, 1), ("YMIN", 1, -1), ("YMAX", 1, 1)):
        nb_ = np.roll(gas_, -off, axis=ax)
        if not (m_pack & nb_).any():
            pf.append(lab)
    # 열린 창은 의도된 환기구라 그대로 재면 실내가 늘 바깥과 통한다.
    # 그래서 창을 전부 닫은 형상으로 '두께를 뺀 실내 공간'을 잰다.
    sealed = build_geometry(replace(c, glass=replace(c.glass, open_windows=())))
    m_seal = rasterize([xb for _, xb, _, _ in sealed["obst"]] + [sealed["pack"][1]], g)
    v_void = int(enclosed_void(m_seal, g).sum()) * g.dx**3
    out = dict(a_frame=a_frame, a_glass=a_glass, a_tire=a_tire, a_pack=a_pack,
               t_body=t_frame, t_frame=t_frame, t_tire=t_tire,
               t_glass=gl.thickness, thr_body_GJ=thr_body,
               dstar=dstar, dstar_over_dx=dstar/g.dx, n_asymmetric=len(asym),
               n_frame_components=n_frame_c, void_m3=v_void, pack_blocked=pf,
               vent_area=sum(v["area"] for v in geo["vents"]))
    if verbose:
        i, j, k = g.ijk
        cab = geo["cabin"]
        print(f"[{c.chid}]")
        print(f"  격자          : {i} x {j} x {k} = {g.ncell:,} cells, dx={g.dx} m")
        print(f"  MPI 메쉬      : {g.nmesh} -> {g.nranks} rank, {g.ncell//g.nranks:,} cells/rank")
        print(f"  D* (7.25 MW)  : {dstar:.2f} m,  D*/dx = {dstar/g.dx:.1f}  (권장 10~16)")
        print(f"  타이어        : 235/45 R18 -> 외경 {ti.diameter*1000:.1f} mm, "
              f"폭 {ti.width*1000:.0f} mm, 축거 {ti.wheelbase:.2f} m")
        print(f"  캐빈          : x {cab[0]:+.3f}~{cab[1]:+.3f}, y {cab[2]:+.3f}~{cab[3]:+.3f}, "
              f"z {cab[4]:.3f}~{cab[5]:.3f}")
        print(f"  노출면적      : 프레임 {a_frame:6.2f} m2  유리 {a_glass:5.2f}  "
              f"타이어 {a_tire:5.2f}  팩 {a_pack:5.2f}")
        print(f"  THICKNESS     : 프레임 {t_frame*1000:6.2f} mm ({m_frame_kg:.0f} kg 역산), "
              f"타이어 {t_tire*1000:6.2f} mm ({m_tire_kg:.0f} kg 역산)")
        print(f"  유리          : {gl.thickness*1000:.1f} mm, "
              f"{'연소' if gl.combustible else '비연소'}, "
              f"열린 창 {gl.open_windows or '없음'} (하단 {gl.open_height*1000:.0f} mm)")
        print(f"  차체 THR      : {thr_body:7.2f} GJ   (논문 Table 4: "
              f"{KANG['targets']['BEV body']['THR_GJ']} GJ)")
        print(f"  벤트 면적     : {out['vent_area']:.3f} m2")
        print(f"  좌우 대칭     : 구조 {'통과' if not asym else str(asym) + ' 비대칭 !!'}"
              f"  |  유리 {len(asym_gl)}개 비대칭 (좌측 창 개방 - 의도)")
        print(f"  프레임 연결   : {n_frame_c}개 성분 "
              f"{'(전부 하나로 연결)' if n_frame_c == 1 else '<- 끊어짐 !!'}")
        P__ = geo["pack"][1]
        zfl = min(b[4] for n, b in
                  {n: xb for n, xb, _, _ in geo["obst"]}.items() if n.startswith("FLOOR"))
        print(f"  팩 6면 노출   : {'6면 전부 기체와 접함' if not pf else str(pf) + ' 막힘 !!'}")
        print(f"  팩 위치       : z {P__[4]:.3f}~{P__[5]:.3f} m  |  "
              f"지상고 {P__[4]:.3f}  바닥({zfl:.3f})과 간격 {zfl - P__[5]:.3f} m")
        print(f"  실내 중공     : {v_void:.3f} m3 (창 닫은 기준) "
              f"{'' if v_void > 0 else '<- 내부가 비어 있지 않다 !!'}")
        for lab, want, got in _snap_report(c, geo):
            print(f"  스냅 {lab:<10}: 요청 {want:6.3f} m -> 격자 {got:6.3f} m "
                  f"({(got-want)*1000:+5.0f} mm, dx={g.dx} m 제약)")
    return out

## 4.1 슬라이스면 · 열유속계 몸체 — 단일 정의

`&SLCF` 위치와 열유속계 몸체 `&OBST` 를 **한 곳에서만 정의**하고 `.fds` 작성과
시각화가 같은 함수를 쓴다. 두 곳에 따로 적으면 반드시 어긋난다.

### 열유속계를 왜 `&OBST` 로 세우는가
FDS에서 `RADIOMETER` 와 `GAUGE HEAT FLUX` 는 **고체상 quantity**라 벽면에 붙어야
한다. 허공에 두면 `ERROR(427) requires repositioning` 으로 죽는다. 실제
Schmidt-Boelter 게이지도 수냉 몸체이므로, 1셀 크기 블록(`TMP_FRONT` = 주변온도)을
세우고 차량을 향한 면에 붙였다.

In [6]:
def slice_planes(c: Case):
    # &SLCF 위치. 시각화와 .fds 작성이 이 함수를 공유한다.
    g, v = c.grid, c.veh
    return [
        dict(axis="x", coord=snap_scalar(0.0, g, 0), note="vehicle centreline (x=0)",
             qty=[("TEMPERATURE", None), ("HRRPUV", None),
                  ("VOLUME FRACTION", "OXYGEN"), ("MASS FRACTION", "BATTERY_GAS")]),
        dict(axis="y", coord=snap_scalar(0.0, g, 1), note="vehicle centre (y=0)",
             qty=[("TEMPERATURE", None), ("HRRPUV", None)]),
        dict(axis="z", coord=snap_scalar(sum(pack_z(c))/2, g, 2),
             note="pack mid-height (under-body vent jets)",
             qty=[("TEMPERATURE", None), ("HRRPUV", None)]),
        dict(axis="z", coord=snap_scalar(Z_PIPE*mm, g, 2),
             note="ceiling TC level z=2.27 m", qty=[("TEMPERATURE", None)]),
    ]


def gauge_bodies(c: Case):
    # 열유속계 수냉 몸체 OBST + 감지점/IOR. 시각화와 .fds 작성이 공유한다.
    g = c.grid
    zw = Z_WIN * mm
    out = []
    for n, x, y, o in SENSORS_HF:
        ax = [i for i, v_ in enumerate(o) if v_ != 0][0]
        sgn = o[ax]
        face = [snap_scalar(px(x), g, 0), snap_scalar(py(y), g, 1), snap_scalar(zw, g, 2)]
        lo, hi = list(face), list(face)
        if sgn > 0: lo[ax] -= g.dx
        else:       hi[ax] += g.dx
        for a in range(3):
            if a != ax:
                lo[a] -= g.dx/2 if a != 2 else 0.0
                hi[a] += g.dx/2 if a != 2 else g.dx
        body = snap_box((lo[0], hi[0], lo[1], hi[1], lo[2], hi[2]), g)
        pt = list(face); pt[ax] += sgn * g.dx / 2       # 감지면에 접한 기체 셀
        out.append(dict(id=n, body=body, xyz=tuple(pt), ior=sgn*(ax+1), orient=o))
    return out


_c0 = Case(chid="probe", title="probe")
for _s in slice_planes(_c0):
    print(f"  SLCF  PB{_s['axis'].upper()} = {_s['coord']:7.4f} m   "
          f"{len(_s['qty'])}개 quantity   ({_s['note']})")
print()
for _b in gauge_bodies(_c0):
    print(f"  {_b['id']:<6} 감지점 {_b['xyz']}  IOR={_b['ior']:+d}  몸체 XB={_b['body']}")

  SLCF  PBX =  1.5500 m   4개 quantity   (vehicle centreline)
  SLCF  PBY =  2.6000 m   2개 quantity   (vehicle centre)
  SLCF  PBZ =  0.2500 m   2개 quantity   (pack mid-height (under-body vent jets))
  SLCF  PBZ =  2.2500 m   1개 quantity   (ceiling TC level z=2.27 m)

  HF_F   감지점 (1.55, 5.1625, 1.25)  IOR=-2  몸체 XB=(1.55, 1.675, 5.225, 5.35, 1.25, 1.375)
  HF_R   감지점 (1.55, 0.0375, 1.25)  IOR=+2  몸체 XB=(1.55, 1.675, -0.15, -0.025, 1.25, 1.375)
  HF_FD  감지점 (0.1125, 2.725, 1.25)  IOR=+1  몸체 XB=(-0.075, 0.05, 2.6, 2.85, 1.25, 1.375)
  HF_RD  감지점 (0.1125, 1.975, 1.25)  IOR=+1  몸체 XB=(-0.075, 0.05, 1.85, 2.1, 1.25, 1.375)


## 5. 인터랙티브 3D — 도메인 안에서 실제로 어떻게 만들어졌는지

Plotly로 **자립형 HTML**을 만든다. 브라우저에서 열면 회전·확대·호버가 되고
Jupyter 커널이 필요 없다 (이 클러스터 로그인 노드에는 jupyter가 없다).
VSCode/Jupyter에서 이 셀을 실행하면 셀 안에서도 바로 돌려볼 수 있다.

**범례를 클릭해서 켜고 끌 수 있다:**

| 범례 항목 | 내용 |
|---|---|
| 차체 / 루프 / 유리 | `SURF_ID='CAR_BODY'`. **루프를 끄면 캐빈 내부가 보인다** |
| 타이어 | `SURF_ID='TIRE'` |
| 배터리 팩 | 비연소 `OBST` + `PACK_CASE` 경계조건 |
| 벤트 Z1/Z2/Z3 | 벤트가스 방출면. Z1이 착화 모듈(Module_9) |
| 열유속계 | 수냉 몸체 + 감지 방향 화살표 |
| 천장 TC / 창문 TC | 논문 센서 위치 |
| 슬라이스면 | `&SLCF` 평면 (기본 꺼짐) |
| 메쉬 경계 | MPI 메쉬 분할면 (기본 꺼짐) |
| 도메인 / 리그 / 지면 | 계산 도메인 외곽, 테스트 리그 외곽 |

호버하면 부재 이름과 **스냅 후 XB 좌표**(= `.fds`에 실제로 쓰이는 값)가 뜬다.

In [7]:
_CUBE_I = [0, 0, 4, 4, 0, 0, 1, 1, 2, 2, 3, 3]
_CUBE_J = [1, 2, 5, 6, 1, 5, 2, 6, 3, 7, 0, 4]
_CUBE_K = [2, 3, 6, 7, 5, 4, 6, 5, 7, 6, 4, 7]


def _cube(xb, thicken=0.0):
    # 두께 0인 축은 thicken 만큼 부풀려 화면에 보이게 한다 (VENT 패치·평면용)
    v = list(xb)
    for a in range(3):
        if abs(v[2*a+1] - v[2*a]) < 1e-12:
            v[2*a] -= thicken; v[2*a+1] += thicken
    x0, x1, y0, y1, z0, z1 = v
    return dict(x=[x0, x1, x1, x0, x0, x1, x1, x0],
                y=[y0, y0, y1, y1, y0, y0, y1, y1],
                z=[z0, z0, z0, z0, z1, z1, z1, z1],
                i=_CUBE_I, j=_CUBE_J, k=_CUBE_K)


def _mesh3d(xb, color, name, group, show_legend, opacity=1.0, visible=True,
            thicken=0.0, hover=None):
    return go.Mesh3d(**_cube(xb, thicken), color=color, opacity=opacity,
                     flatshading=True, name=name, legendgroup=group,
                     showlegend=show_legend, visible=visible,
                     hovertemplate=(hover or name) + "<extra></extra>")


def _plane(axis, coord, g: Grid, color, name, group, show_legend, opacity=0.20,
           visible="legendonly"):
    lim = dict(x=(g.xmin, g.xmax), y=(g.ymin, g.ymax), z=(g.zmin, g.zmax))
    xb = {"x": (coord, coord, *lim["y"], *lim["z"]),
          "y": (*lim["x"], coord, coord, *lim["z"]),
          "z": (*lim["x"], *lim["y"], coord, coord)}[axis]
    return _mesh3d(xb, color, name, group, show_legend, opacity=opacity,
                   visible=visible, thicken=1e-3,
                   hover=f"{name}<br>PB{axis.upper()} = {coord:.4f} m")


PLOT_STYLE = {
    "body":  ("#9fb0c4", "프레임 (CAR_BODY)"),
    "glass": ("#7fc6e8", "유리 (CAR_GLASS)"),
    "roof":  ("#c8d2dd", "루프  ← 끄면 캐빈 내부"),
    "tire":  ("#2b2b2b", "타이어 (TIRE)"),
    "pack":  ("#e08a2b", "배터리 팩 (비연소)"),
    "gauge": ("#2b57c4", "열유속계 수냉 몸체"),
}


def preview_plotly(c: Case, save=None):
    if not HAVE_PLOTLY:
        print("plotly 미설치 - 인터랙티브 3D 건너뜀"); return None
    geo, g, v = build_geometry(c), c.grid, c.veh
    tr, seen = [], set()

    def kind(name, surf):
        if surf == SURF_TIRE:  return "tire"
        if surf == SURF_GLASS: return "glass"
        if name == "ROOF":     return "roof"
        if name.startswith("GLASS_"): return "glass"
        return "body"

    for n, xb, surf, _ in geo["obst"]:
        kd = kind(n, surf)
        col, lab = PLOT_STYLE[kd]
        hv = (f"<b>{n}</b><br>SURF_ID={surf}<br>XB = {fmt_xb(xb)}<br>"
              f"크기 {xb[1]-xb[0]:.3f} x {xb[3]-xb[2]:.3f} x {xb[5]-xb[4]:.3f} m")
        tr.append(_mesh3d(xb, col, lab, kd, kd not in seen, hover=hv))
        seen.add(kd)

    P = geo["pack"][1]
    tr.append(_mesh3d(P, PLOT_STYLE["pack"][0], PLOT_STYLE["pack"][1], "pack", True,
                      hover=f"<b>PACK</b> (비연소, SURF_ID=PACK_CASE)<br>XB = {fmt_xb(P)}"))

    vcol = {"PACK": "#e02020", "Z1": "#e02020", "Z2": "#f2801a", "Z3": "#c21b8a"}
    for vz in geo["vents"]:
        lab = f"벤트가스 방출면 (팩 {len(vz['patches'])}면)"
        first = True
        for fn, q in zip(vz.get("faces", vz["patches"]), vz["patches"]):
            tr.append(_mesh3d(q, vcol.get(vz["name"], "#e02020"), lab,
                              f"vent{vz['name']}", first, thicken=g.dx*0.08,
                              hover=(f"<b>VENT_{vz['name']}</b> ({fn})<br>XB = {fmt_xb(q)}"
                                     f"<br>6면 합 면적 {vz['area']:.4f} m2")))
            first = False

    gb = gauge_bodies(c)
    for i, b_ in enumerate(gb):
        tr.append(_mesh3d(b_["body"], PLOT_STYLE["gauge"][0], PLOT_STYLE["gauge"][1],
                          "gauge", i == 0,
                          hover=f"<b>{b_['id']}</b> 수냉 몸체<br>XB = {fmt_xb(b_['body'])}"))
    tr.append(go.Scatter3d(
        x=[b_["xyz"][0] for b_ in gb], y=[b_["xyz"][1] for b_ in gb],
        z=[b_["xyz"][2] for b_ in gb], mode="markers+text",
        marker=dict(size=6, color="#2b57c4", symbol="diamond"),
        text=[b_["id"] for b_ in gb], textposition="top center",
        textfont=dict(size=10, color="#2b57c4"),
        name="열유속계 감지점 (z=1.27 m)", legendgroup="gauge", showlegend=True,
        hovertemplate="<b>%{text}</b><br>XYZ = (%{x:.3f}, %{y:.3f}, %{z:.3f})<extra></extra>"))
    for b_ in gb:
        o, p0, Lq = b_["orient"], b_["xyz"], 0.6
        tr.append(go.Scatter3d(
            x=[p0[0], p0[0]+o[0]*Lq], y=[p0[1], p0[1]+o[1]*Lq], z=[p0[2], p0[2]+o[2]*Lq],
            mode="lines", line=dict(color="#2b57c4", width=6),
            name="감지 방향", legendgroup="gauge", showlegend=False, hoverinfo="skip"))

    zp, zw = Z_PIPE*mm, Z_WIN*mm
    tr.append(go.Scatter3d(
        x=[px(x) for _, x, _ in SENSORS_TC_CEIL], y=[py(y) for _, _, y in SENSORS_TC_CEIL],
        z=[zp]*len(SENSORS_TC_CEIL), mode="markers+text",
        marker=dict(size=6, color="#d81919"),
        text=[n for n, _, _ in SENSORS_TC_CEIL], textposition="top center",
        textfont=dict(size=10, color="#d81919"),
        name=f"천장 열전대 6개 (z={zp:.3f} m)", legendgroup="tcceil",
        hovertemplate="<b>%{text}</b><br>XYZ = (%{x:.3f}, %{y:.3f}, %{z:.3f})<extra></extra>"))
    tr.append(go.Scatter3d(
        x=[px(x) for _, x, _ in SENSORS_TC_WIN], y=[py(y) for _, _, y in SENSORS_TC_WIN],
        z=[zw]*len(SENSORS_TC_WIN), mode="markers+text",
        marker=dict(size=7, color="#8b0000", symbol="square"),
        text=[n for n, _, _ in SENSORS_TC_WIN], textposition="bottom center",
        textfont=dict(size=10, color="#8b0000"),
        name=f"창문 열전대 2개 (z={zw:.3f} m)", legendgroup="tcwin",
        hovertemplate="<b>%{text}</b><br>XYZ = (%{x:.3f}, %{y:.3f}, %{z:.3f})<extra></extra>"))

    for sp in slice_planes(c):
        tr.append(_plane(sp["axis"], sp["coord"], g, "#2e8b57",
                         f"SLCF PB{sp['axis'].upper()}={sp['coord']:.3f}  ({sp['note']})",
                         "slcf", True, opacity=0.18))
    _, planes = mesh_lines(g)
    first = True
    for ax_ in "xyz":
        for q in planes[ax_]:
            tr.append(_plane(ax_, q, g, "#ff8c00",
                             f"MPI 메쉬 경계  {ax_} = {q:.3f} m", "mesh", first, opacity=0.15))
            first = False

    X, Y, Z = (g.xmin, g.xmax), (g.ymin, g.ymax), (g.zmin, g.zmax)
    ex, ey, ez = [], [], []
    for a in X:
        for b in Y:
            ex += [a, a, None]; ey += [b, b, None]; ez += [Z[0], Z[1], None]
    for k_ in Z:
        ex += [X[0], X[1], X[1], X[0], X[0], None]
        ey += [Y[0], Y[0], Y[1], Y[1], Y[0], None]
        ez += [k_]*5 + [None]
    tr.append(go.Scatter3d(x=ex, y=ey, z=ez, mode="lines",
                           line=dict(color="black", width=2),
                           name=(f"계산 도메인 {X[1]-X[0]:.1f} x {Y[1]-Y[0]:.1f} x "
                                 f"{Z[1]-Z[0]:.1f} m (측·상면 OPEN)"),
                           legendgroup="dom", hoverinfo="skip"))
    # 리그도 논문 좌표 그대로 두면 차량만 원점으로 옮겨져 어긋난다.
    RW, RL = RIG_W*mm, RIG_L*mm
    rx0, rx1 = px(0.0), px(RIG_W)
    ry0, ry1 = py(0.0), py(RIG_L)
    tr.append(go.Scatter3d(x=[rx0, rx1, rx1, rx0, rx0],
                           y=[ry0, ry0, ry1, ry1, ry0], z=[0.01]*5,
                           mode="lines", line=dict(color="black", width=5, dash="dash"),
                           name=f"테스트 리그 {RW:.2f} x {RL:.2f} m", legendgroup="dom",
                           hoverinfo="skip"))
    tr.append(_mesh3d((g.xmin, g.xmax, g.ymin, g.ymax, -g.dx, 0.0), "#d9d9d9",
                      "지면 (ZMIN = 고체)", "dom", True, opacity=0.45,
                      hover="지면: &VENT MB='ZMIN', SURF_ID='GROUND'"))

    i, j, k = g.ijk
    fig = go.Figure(tr)
    fig.update_layout(
        title=dict(text=(f"<b>{c.chid}</b> — Kang et al. (2023) Test 4 재현 &nbsp;|&nbsp; "
                         f"도메인 {X[1]-X[0]:.1f} x {Y[1]-Y[0]:.1f} x {Z[1]-Z[0]:.1f} m = "
                         f"{i}x{j}x{k} = {g.ncell:,} cells (dx={g.dx} m), "
                         f"{g.nranks} MPI mesh<br>"
                         f"<sub>좌표는 전부 격자 스냅 후 값 = .fds 에 실제로 쓰이는 값. "
                         f"범례 클릭으로 켜고 끄기 (루프를 끄면 캐빈 내부). "
                         f"x=폭, y=길이(+y 전방), z=플랫폼 상면 기준</sub>"),
                   x=0.01, font=dict(size=13)),
        scene=dict(xaxis_title="x [m]  (width)",
                   yaxis_title="y [m]  (length, +y = front)",
                   zaxis_title="z [m]", aspectmode="data",
                   camera=dict(eye=dict(x=1.7, y=-1.9, z=1.15))),
        legend=dict(itemsizing="constant", font=dict(size=11), y=0.98),
        margin=dict(l=0, r=0, t=95, b=0), height=830)
    if save:
        fig.write_html(str(save), include_plotlyjs=True, full_html=True)
    return fig


FIG = preview_plotly(Case(chid="KANG_T4_preview", title="preview"))
print("figure trace 수:", len(FIG.data) if FIG else 0)

plotly 미설치 - 인터랙티브 3D 건너뜀
figure trace 수: 0


### 5.1 노트북 셀 안에서 바로 인터랙티브로 보기

HTML 파일을 열지 않고 **이 셀 출력 안에서** 회전·확대·호버가 된다.
아래 셀이 환경에 맞는 Plotly 렌더러를 골라 인라인으로 띄운다.

**필요 조건 (이미 설치해 둠):** `plotly`, `nbformat`, `ipykernel`, `ipywidgets`,
`anywidget`. VSCode 에서는 우측 상단 **Select Kernel → Python 3.9 (FDS)**
(또는 `/usr/bin/python3`) 를 고르면 된다.

Plotly 기본 렌더러가 `browser` 라서 원격 노드에서는 아무것도 안 뜬다.
그래서 `vscode` / `notebook_connected` 를 우선 시도한다.

`FigureWidget` 로 감싸면 파이썬에서 트레이스를 켜고 끄는 것도 된다:

```python
W = go.FigureWidget(FIG)
W                                   # 위젯으로 표시
W.data[0].visible = False           # 파이썬 코드로 부재 숨기기
for t in W.data:                    # 루프만 남기고 다 끄기
    t.visible = (t.name or "").startswith("루프")
```

In [8]:
import plotly.io as pio


def in_ipython_kernel():
    # run_notebook.py(헤드리스)에서는 인라인 표시를 건너뛰기 위한 판별
    try:
        from IPython import get_ipython
        ip = get_ipython()
        return ip is not None and ip.__class__.__name__ == "ZMQInteractiveShell"
    except Exception:
        return False


def show_inline(fig, prefer=("vscode", "notebook_connected", "notebook",
                             "jupyterlab", "plotly_mimetype", "iframe_connected")):
    # 환경에 맞는 렌더러를 골라 셀 출력 안에 인라인으로 띄운다.
    # Plotly 기본값은 'browser' 라서 원격/헤드리스에서는 아무것도 안 보인다.
    if fig is None:
        print("figure 없음 (plotly 미설치?)"); return
    if not in_ipython_kernel():
        print("IPython 커널이 아님 -> 인라인 표시 생략.")
        print("  VSCode/Jupyter 에서 이 노트북을 열고 커널을 붙여 실행하면 여기 그림이 뜬다.")
        print("  헤드리스로는 <CHID>_geometry.html 을 브라우저에서 열면 된다.")
        return
    avail = set(pio.renderers)
    for r in prefer:
        if r in avail:
            try:
                fig.show(renderer=r)
                print(f"[renderer] {r}")
                return
            except Exception as e:
                print(f"[renderer] {r} 실패: {type(e).__name__} {e}")
    print("사용 가능한 인라인 렌더러가 없다. 렌더러 목록:", sorted(avail))


show_inline(FIG)

ModuleNotFoundError: No module named 'plotly'

### 5.2 빠른 확인 루프 — `quick_check()`

**FDS를 돌리기 전에** 파라미터를 바꿔가며 형상을 즉시 확인하는 용도. 한 줄로
`.fds` 를 쓰지도, HTML 을 굽지도 않고 형상 표 + 인터랙티브 3D 만 띄운다.

```python
quick_check()                                  # 현재 기본값
quick_check(dx=0.10)                           # 격자만 바꿔서
quick_check(pack_len=2.4, pack_wid=1.5)        # 팩 크기
quick_check(z_roof=1.60, z_belt=0.95)          # 차량 프로파일
quick_check(body=dict(heat_of_reaction=2500))  # 이름이 겹치면 그룹을 명시
quick_check(show=False)                        # 표만 (3D 생략)
```

필드 이름만 주면 어느 dataclass 로 갈지 자동으로 찾는다. `soot_yield` 처럼
`BatteryGas`/`CarBody` 양쪽에 있는 이름은 모호하므로 그룹을 명시하라고 알려준다.

In [ ]:
from dataclasses import fields as _dc_fields

_GROUPS = dict(grid=Grid, veh=Vehicle, tire=Tire, glass=Glass, frame=Frame,
               pack=Pack, vent=VentLayout, gas=BatteryGas, hrr=PackHRR)
_CASE_FIELDS = {f.name for f in _dc_fields(Case)} - set(_GROUPS)


def make_case(chid="QUICK", title="quick check", **over):
    # 필드 이름만 주면 어느 dataclass 소속인지 찾아 넣는다.
    # 그룹을 명시하려면 grid=dict(...), body=dict(...) 처럼 쓴다.
    kw = {g: dict(over.pop(g, {})) for g in _GROUPS}
    top = {}
    for k, v in over.items():
        if k in _CASE_FIELDS:
            top[k] = v; continue
        owners = [g for g, cls in _GROUPS.items() if k in {f.name for f in _dc_fields(cls)}]
        if not owners:
            raise KeyError(f"'{k}' 라는 파라미터가 없다. "
                           f"Case/{'/'.join(_GROUPS)} 의 필드를 확인할 것")
        if len(owners) > 1:
            raise KeyError(f"'{k}' 는 {owners} 에 모두 있어 모호하다. "
                           f"예: {owners[0]}=dict({k}=...) 처럼 그룹을 명시할 것")
        kw[owners[0]][k] = v
    c = Case(chid=chid, title=title, **top)
    for g, d in kw.items():
        if d:
            c = replace(c, **{g: replace(getattr(c, g), **d)})
    return c


def obst_table(c: Case, geo=None):
    # .fds 에 실제로 쓰이는 (격자 스냅 후) 좌표 표
    geo = geo or build_geometry(c)
    rows = []
    for n, xb, surf, _ in geo["obst"]:
        rows.append(dict(name=n, surf=surf, x0=xb[0], x1=xb[1], y0=xb[2], y1=xb[3],
                         z0=xb[4], z1=xb[5],
                         dx=xb[1]-xb[0], dy=xb[3]-xb[2], dz=xb[5]-xb[4]))
    P = geo["pack"][1]
    rows.append(dict(name="PACK", surf="PACK_CASE", x0=P[0], x1=P[1], y0=P[2], y1=P[3],
                     z0=P[4], z1=P[5], dx=P[1]-P[0], dy=P[3]-P[2], dz=P[5]-P[4]))
    for vz in geo["vents"]:
        for i, q in enumerate(vz["patches"], 1):
            rows.append(dict(name=f"VENT_{vz['name']}_{i}", surf="(VENT)",
                             x0=q[0], x1=q[1], y0=q[2], y1=q[3], z0=q[4], z1=q[5],
                             dx=q[1]-q[0], dy=q[3]-q[2], dz=q[5]-q[4]))
    for b_ in gauge_bodies(c):
        q = b_["body"]
        rows.append(dict(name=f"BODY_{b_['id']}", surf="HF_GAUGE",
                         x0=q[0], x1=q[1], y0=q[2], y1=q[3], z0=q[4], z1=q[5],
                         dx=q[1]-q[0], dy=q[3]-q[2], dz=q[5]-q[4]))
    return pd.DataFrame(rows).set_index("name")


def quick_check(show=True, table=True, **over):
    # FDS 를 돌리기 전 형상 확인용. .fds 도 HTML 도 만들지 않는다.
    c = make_case(**over)
    geo = build_geometry(c)
    qa = geometry_qa(c, geo, verbose=True)
    if table:
        print()
        print(obst_table(c, geo).to_string(
            float_format=lambda v: f"{v:7.3f}",
            columns=["surf", "x0", "x1", "y0", "y1", "z0", "z1", "dx", "dy", "dz"]))
    if show:
        show_inline(preview_plotly(c))
    return c


_ = quick_check(show=True)

## 5.4 정적 그림 — 도메인 전체 단면 (보고서용)

확대 없이 도메인 끝까지 그린다. 플룸이 올라갈 높이가 충분한지, 개방 경계가
화염에서 얼마나 떨어져 있는지, 메쉬 경계면과 슬라이스면이 어디를 지나는지를
한눈에 본다. 단면 위치는 `slice_planes()` 와 같은 값이다.

In [ ]:
PART_COLOR = {"CAR_BODY": "#b9c2cc", "CAR_GLASS": "#7fc6e8",
              "TIRE": "#2b2b2b", "PACK_CASE": "#c8792b"}


def preview_domain(c: Case, save=None):
    geo, g, v = build_geometry(c), c.grid, c.veh
    grp = {"CAR_BODY": [], "CAR_GLASS": [], "TIRE": [], "PACK_CASE": [geo["pack"][1]]}
    for _, xb, s_, _ in geo["obst"]:
        grp[s_].append(xb)
    masks = {k: rasterize(b, g) for k, b in grp.items() if b}
    # 두께 0 인 VENT 패치를 화면에 보이게 부풀린다. 반드시 '팩 안쪽'으로
    # 부풀려야 한다. 바깥으로 부풀리면 팩과 차체 바닥 사이의 1셀 간격을
    # 덮어버려 붙어 있는 것처럼 보인다 (6면 벤트로 바꾼 뒤 실제로 그랬다).
    Pk = geo["pack"][1]
    vpat = []
    for vz in geo["vents"]:
        for q in vz["patches"]:
            w = list(q)
            for a in range(3):
                if abs(w[2*a+1] - w[2*a]) > 1e-12:
                    continue
                if abs(w[2*a] - Pk[2*a+1]) < 1e-9:      # 상/우/전면 -> 안쪽은 -
                    w[2*a] -= g.dx
                else:                                    # 하/좌/후면 -> 안쪽은 +
                    w[2*a+1] += g.dx
            vpat.append(tuple(w))
    mvent = rasterize(vpat, g)
    mgauge = rasterize([b_["body"] for b_ in gauge_bodies(c)], g)

    X = np.linspace(g.xmin, g.xmax, g.ijk[0]+1)
    Y = np.linspace(g.ymin, g.ymax, g.ijk[1]+1)
    Z = np.linspace(g.zmin, g.zmax, g.ijk[2]+1)
    sl = slice_planes(c)
    ic = int(round((sl[0]["coord"]-g.xmin)/g.dx))
    jc = int(round((sl[1]["coord"]-g.ymin)/g.dx))
    kc = int(round((sl[2]["coord"]-g.zmin)/g.dx))
    _, planes = mesh_lines(g)
    sx = [s["coord"] for s in sl if s["axis"] == "x"]
    sy = [s["coord"] for s in sl if s["axis"] == "y"]
    sz = [s["coord"] for s in sl if s["axis"] == "z"]

    fig, ax = plt.subplots(1, 3, figsize=(17, 5.8))
    views = [(0, f"y-z  section at x = {X[ic]:.3f} m (vehicle centreline)", Y, Z,
              lambda m: m[ic, :, :], "y [m]", "z [m]", sy, sz, planes["y"], planes["z"]),
             (1, f"x-y  section at z = {Z[kc]:.3f} m (pack mid-height)", X, Y,
              lambda m: m[:, :, kc], "x [m]", "y [m]", sx, sy, planes["x"], planes["y"]),
             (2, f"x-z  section at y = {Y[jc]:.3f} m (vehicle centre)", X, Z,
              lambda m: m[:, jc, :], "x [m]", "z [m]", sx, sz, planes["x"], planes["z"])]
    for i, title, A, B, sel, xl, yl, pa, pb, ma, mb in views:
        for k, m in masks.items():
            f = np.where(sel(m), 1., np.nan)
            ax[i].pcolormesh(A, B, f.T, cmap=matplotlib.colors.ListedColormap([PART_COLOR[k]]),
                             vmin=0, vmax=1, shading="flat")
        for msk, col in ((mgauge, "#2b57c4"), (mvent, "#e03030")):
            f = np.where(sel(msk), 1., np.nan)
            ax[i].pcolormesh(A, B, f.T, cmap=matplotlib.colors.ListedColormap([col]),
                             vmin=0, vmax=1, shading="flat")
        for q in ma: ax[i].axvline(q, color="darkorange", lw=1.4, ls="--", alpha=.95)
        for q in mb: ax[i].axhline(q, color="darkorange", lw=1.4, ls="--", alpha=.95)
        for q in pa: ax[i].axvline(q, color="seagreen", lw=1.2, ls=":", alpha=.95)
        for q in pb: ax[i].axhline(q, color="seagreen", lw=1.2, ls=":", alpha=.95)
        ax[i].set_xlim(A[0], A[-1]); ax[i].set_ylim(B[0], B[-1])
        ax[i].set_title(title, fontsize=9)
        ax[i].set_xlabel(xl); ax[i].set_ylabel(yl)
        ax[i].set_aspect("equal"); ax[i].grid(alpha=.18, lw=.3)
        for spine in ax[i].spines.values():
            spine.set_edgecolor("tab:cyan"); spine.set_linewidth(2.2)
    ax[0].axhline(0, color="dimgray", lw=3)
    ax[2].axhline(0, color="dimgray", lw=3)
    fig.suptitle(f"{c.chid}  full-domain sections  —  cyan frame = OPEN boundary, "
                 f"grey bottom = solid ground, orange dashed = MPI mesh boundary, "
                 f"green dotted = SLCF plane, red = VENT patch, blue = heat-flux gauge",
                 fontsize=9)
    fig.tight_layout()
    if save: fig.savefig(save, dpi=145, bbox_inches="tight")
    return fig


preview_domain(Case(chid="preview", title="preview"))

## 6. 열화학 · 1D 인터페이스 (앞 노트북과 동일 규약)

In [ ]:
SPECIES_DB = {
    "H2":   ("HYDROGEN",         2.016, 120.0),
    "CO":   ("CARBON MONOXIDE", 28.010,  10.1),
    "CO2":  ("CARBON DIOXIDE",  44.010,   0.0),
    "CH4":  ("METHANE",         16.043,  50.0),
    "C2H4": ("ETHYLENE",        28.054,  47.2),
    "HF":   ("HYDROGEN FLUORIDE", 20.006, 0.0),
}
FDS_ID = {k: v[0] for k, v in SPECIES_DB.items()}
MW  = {k: v[1] for k, v in SPECIES_DB.items()}
DHC = {k: v[2] for k, v in SPECIES_DB.items()}


def thermochem(gas: BatteryGas):
    x = {k: v/sum(gas.composition.values()) for k, v in gas.composition.items()}
    w = sum(x[k]*MW[k] for k in x)
    Y = {k: x[k]*MW[k]/w for k in x}
    return dict(x=x, Y=Y, W_mix=w, dhc_mix=sum(Y[k]*DHC[k] for k in Y)*1000.)


def rdp_mask(t, f, eps):
    t = np.asarray(t, float); f = np.asarray(f, float)
    keep = np.zeros(len(t), bool); keep[0] = keep[-1] = True
    st = [(0, len(t)-1)]
    while st:
        i, j = st.pop()
        if j <= i+1: continue
        d = np.abs(f[i:j+1] - np.interp(t[i:j+1], [t[i], t[j]], [f[i], f[j]]))
        k = int(np.argmax(d))
        if d[k] > eps:
            k += i; keep[k] = True; st += [(i, k), (k, j)]
    return keep


def make_ramp(rid, t, f, eps, prec=3):
    """RAMP 줄 생성. T 는 반드시 순증가여야 한다 (FDS ERROR(394)).

    디지타이즈 곡선에는 같은 시각에 두 점이 찍힌 수직 구간이 있다
    (pack_HRR.csv 는 16.35514 min 에서 간격이 2e-13 s 다). 원본은 오름차순이라도
    소수점으로 형식화하면 같은 값이 되어 FDS 가 입력을 거부한다. 뒤 점을 버리면
    수직 상승이 사라지므로, 형식 최소단위만큼 밀어 두 점을 모두 살린다.
    """
    out, last = [], None
    for i in np.flatnonzero(rdp_mask(t, f, eps)):
        ts = round(float(t[i]), prec)
        if last is not None and ts <= last:
            ts = round(last + 10.0**-prec, prec)
        out.append(f"&RAMP ID='{rid}', T={ts:10.{prec}f}, F={f[i]:7.5f} /")
        last = ts
    return out


def load_1d(n=3):
    if not (DATA_1D / "vent_zone1.csv").exists():
        subprocess.run(["python3", str(DATA_1D/"make_dummy_1d_data.py")], check=True)
    return ([pd.read_csv(DATA_1D/f"vent_zone{i+1}.csv") for i in range(n)],
            pd.read_csv(DATA_1D/"casing_heat_flux.csv"))


def load_pack_hrr(cfg: PackHRR):
    """ref_data csv -> (t[s], HRR[kW]) 와 선형보간 함수.

    0번째 열이 오름차순인지 반드시 확인한다. 어긋나 있으면 np.interp 가
    조용히 틀린 값을 준다 (경고도 안 띄운다).
    """
    d = np.loadtxt(REF / cfg.csv, delimiter=",")
    t, q = d[:, 0].astype(float), d[:, 1].astype(float)
    bad = np.flatnonzero(np.diff(t) <= 0)
    if bad.size:
        raise ValueError(f"{cfg.csv}: 0번째 열이 오름차순이 아니다 "
                         f"({bad.size}곳, 첫 위치 idx {bad[0]}: "
                         f"{t[bad[0]]:.4f} -> {t[bad[0]+1]:.4f})")
    n_neg = int((q < 0).sum())
    dt_min = float(np.diff(t).min())
    if cfg.clip_negative:
        q = np.clip(q, 0.0, None)
    t_s  = t * cfg.t_unit
    q_kW = q * cfg.q_unit * cfg.scale
    f = lambda tq: np.interp(tq, t_s, q_kW, left=q_kW[0], right=q_kW[-1])
    return dict(t_s=t_s, q_kW=q_kW, f=f, n_negative=n_neg, dt_min_s=dt_min*cfg.t_unit,
                thr_GJ=float(_trapz(q_kW, t_s)) / 1e6,
                peak_MW=float(q_kW.max()) / 1e3,
                t_peak_min=float(t_s[int(np.argmax(q_kW))]) / 60.0)


ZONES_1D, CASING_1D = load_1d()
PACK_HRR = load_pack_hrr(PackHRR())
TC0 = thermochem(BatteryGas())
_m = sum(_trapz(z.mdot_kg_s, z.t_s) for z in ZONES_1D)
print(f"[팩 HRR - ref_data/{PackHRR().csv}]")
print(f"  구간          : 0 ~ {PACK_HRR['t_s'][-1]/60:.1f} min "
      f"({len(PACK_HRR['t_s'])}점, 0번째 열 오름차순 확인)")
print(f"  pHRR          : {PACK_HRR['peak_MW']:.2f} MW @ {PACK_HRR['t_peak_min']:.1f} min"
      f"   (논문 Table 4 팩 {KANG['targets']['LIB pack']['pHRR_MW']} MW, "
      f"{KANG['t_peak_min'][0]:.0f}~{KANG['t_peak_min'][1]:.0f} min)")
print(f"  THR (적분)    : {PACK_HRR['thr_GJ']:.2f} GJ"
      f"   (논문 Table 4 팩 {KANG['targets']['LIB pack']['THR_GJ']} GJ)")
if PACK_HRR["dt_min_s"] < 1e-3:
    print(f"  ! 최소 시간간격 {PACK_HRR['dt_min_s']:.2e} s - 사실상 중복 시각(수직 구간)."
          f" make_ramp 가 형식 최소단위만큼 밀어 순증가를 보장한다")
if PACK_HRR["n_negative"]:
    print(f"  ! 음수 HRR {PACK_HRR['n_negative']}점 -> 0 으로 클립 "
          f"(디지타이즈 잡음. 음수면 FDS 로 음의 질량유속이 나간다)")
print()
print(f"벤트가스 dHc      : {TC0['dhc_mix']:.0f} kJ/kg")
print(f"1D 방출질량 합    : {_m:.1f} kg   (논문 Table 6 팩 소실질량 "
      f"{KANG['targets']['LIB pack']['dm']:.0f} kg)")
print(f"1D 기준 팩 THR    : {_m*TC0['dhc_mix']/1e6:.2f} GJ  "
      f"(논문 Table 4 {KANG['targets']['LIB pack']['THR_GJ']} GJ)")
print()
print("※ 논문 팩 dHc,eff = 45.9 MJ/kg 은 벤트가스(14.6)보다 훨씬 크다.")
print("  분리막·전해액 등 고체 가연분 연소가 포함된 값이므로, 팩 기여를 벤트가스")
print("  하나로만 모사하면 THR 이 부족하다. 1D 실데이터 도착 시 반드시 재설계.")

## 7. FDS 입력 생성 + 배치 검사

In [ ]:
HDR = lambda s: f"\n! {'='*72}\n!  {s}\n! {'='*72}"


def write_fds(c: Case, outdir: Path) -> Path:
    g, v, b = c.grid, c.veh, c.frame
    geo = build_geometry(c); qa = geometry_qa(c, geo, verbose=False)
    tc = thermochem(c.gas); hoc = c.gas.heat_of_combustion or tc["dhc_mix"]
    outdir.mkdir(parents=True, exist_ok=True)
    L = [f"&HEAD CHID='{c.chid}', TITLE='{c.title}' /",
         "! 생성: input/build_kang_validation.ipynb (수동 편집 금지)",
         "! 검증 대상: Kang et al., Applied Energy 332 (2023) 120497, Test 4 (BEV_3, 64 kWh)",
         "! 좌표계: x=폭, y=길이(후->전), z=플랫폼 상면 기준.",
         f"!         차량 중심이 원점 (0,0). 논문 좌표에서 "
         f"({-PAPER_ORIGIN[0]:+.3f}, {-PAPER_ORIGIN[1]:+.3f}) m 평행이동."]

    L.append(HDR("메쉬 · 시간 · 출력"))
    ml, planes = mesh_lines(g); L += ml
    L.append(f"! 메쉬 경계면: x={planes['x']}, y={planes['y']}, z={planes['z']}")
    L.append(f"&TIME T_END={c.t_end:.1f} /")
    L.append(f"&MISC TMPA={c.tmpa:.1f} /")
    L.append(f"&DUMP DT_HRR={c.dt_hrr}, DT_DEVC={c.dt_devc}, DT_SLCF={c.dt_slcf}, "
             f"DT_BNDF={c.dt_bndf}, MASS_FILE=T /   ! MASS_FILE = 논문 로드셀 대응")

    L.append(HDR("화학종"))
    for k in c.gas.composition:
        L.append(f"&SPEC ID={chr(39)+FDS_ID[k]+chr(39):<19} LUMPED_COMPONENT_ONLY=T /")
    parts = ["&SPEC ID='BATTERY_GAS'"] + [
        f"      SPEC_ID({n})='{FDS_ID[k]}', VOLUME_FRACTION({n})={c.gas.composition[k]:.3f}"
        for n, k in enumerate(c.gas.composition, 1)]
    L.append(",\n".join(parts) + " /")
    L.append(f"&SPEC ID='CAR_BODY_FUEL', FORMULA='{b.formula}' /")
    if c.gas.track_hf:
        L.append(f"&SPEC ID='{FDS_ID['HF']}' /   ! 비반응 추적")

    L.append(HDR("반응"))
    L.append(f"&REAC ID='BATT', FUEL='BATTERY_GAS', HEAT_OF_COMBUSTION={hoc:.1f},")
    L.append(f"      SOOT_YIELD={c.gas.soot_yield}, CO_YIELD={c.gas.co_yield}, "
             f"RADIATIVE_FRACTION={c.gas.radiative_fraction} /")
    L.append(f"&REAC ID='BODY', FUEL='CAR_BODY_FUEL', "
             f"HEAT_OF_COMBUSTION={b.heat_of_combustion:.1f},   ! Table 6 실측")
    L.append(f"      SOOT_YIELD={b.soot_yield}, CO_YIELD={b.co_yield}, "
             f"RADIATIVE_FRACTION={b.radiative_fraction} /")

    ti, gl = c.tire, c.glass
    L.append(HDR("부품별 재료 - 프레임 / 타이어 / 유리 를 독립 파라미터로 둔다"))
    L.append("! 프레임과 타이어는 열적 물성만 다르고 연료 화학종은 CAR_BODY_FUEL 로")
    L.append("! 공유한다 (REAC 를 늘리지 않기 위해서). THICKNESS 는 각자 질량에서 역산.")
    L.append("&MATL ID='FRAME_MATL'\n"
             "      SPEC_ID='CAR_BODY_FUEL'\n"
             "      NU_SPEC=1.0\n"
             f"      BOILING_TEMPERATURE={b.boiling_temperature:.1f}\n"
             f"      HEAT_OF_REACTION={b.heat_of_reaction:.1f}\n"
             f"      DENSITY={b.density:.1f}\n"
             f"      CONDUCTIVITY={b.conductivity}\n"
             f"      SPECIFIC_HEAT={b.specific_heat}\n"
             f"      ABSORPTION_COEFFICIENT={b.absorption_coefficient:.1f}\n"
             f"      EMISSIVITY={b.emissivity} /")
    L.append("&MATL ID='TIRE_MATL'\n"
             "      SPEC_ID='CAR_BODY_FUEL'\n"
             "      NU_SPEC=1.0\n"
             f"      BOILING_TEMPERATURE={ti.boiling_temperature:.1f}\n"
             f"      HEAT_OF_REACTION={ti.heat_of_reaction:.1f}\n"
             f"      DENSITY={ti.density:.1f}\n"
             f"      CONDUCTIVITY={ti.conductivity}\n"
             f"      SPECIFIC_HEAT={ti.specific_heat}\n"
             f"      ABSORPTION_COEFFICIENT={ti.absorption_coefficient:.1f}\n"
             f"      EMISSIVITY={ti.emissivity} /")
    L.append("! 유리는 비연소. 형상에서 빼지 않고 이 SURF 로 채워 두었으므로,")
    L.append("! 물성을 바꾸면 캐빈 열적 거동만 독립으로 손볼 수 있다.")
    L.append("&MATL ID='GLASS_MATL'\n"
             f"      DENSITY={gl.density:.1f}\n"
             f"      CONDUCTIVITY={gl.conductivity}\n"
             f"      SPECIFIC_HEAT={gl.specific_heat}\n"
             f"      EMISSIVITY={gl.emissivity} /")
    L.append(f"&SURF ID='CAR_BODY',  MATL_ID='FRAME_MATL', "
             f"THICKNESS={qa['t_frame']:.5f}, COLOR='SILVER' /")
    L.append(f"&SURF ID='TIRE',      MATL_ID='TIRE_MATL',  "
             f"THICKNESS={qa['t_tire']:.5f}, COLOR='BLACK' /")
    L.append(f"&SURF ID='CAR_GLASS', MATL_ID='GLASS_MATL', "
             f"THICKNESS={gl.thickness:.5f}, COLOR='SKY BLUE' /")
    L.append("&SURF ID='GROUND', COLOR='GRAY 40' /")
    L.append("! Schmidt-Boelter 게이지는 수냉식이라 표면이 상온에 가깝게 유지된다.")
    L.append(f"&SURF ID='HF_GAUGE', TMP_FRONT={c.tmpa:.1f}, COLOR='BLUE' /")

    L.append(HDR("경로 1 - 팩 케이싱 고체 경계조건"))
    qp = float(CASING_1D.q_net_kW_m2.max())
    L.append(f"&SURF ID='PACK_CASE', COLOR='GRAY 30', NET_HEAT_FLUX={qp:.4f}, RAMP_Q='cas_q' /")
    L += make_ramp("cas_q", CASING_1D.t_s.values, (CASING_1D.q_net_kW_m2/qp).values, c.ramp_eps)

    L.append(HDR("경로 2 - 팩 발열률. ref_data 실측곡선을 팩 6면에 면적비례로"))
    vz = geo["vents"][0]
    ph = load_pack_hrr(c.hrr)
    A  = vz["area"]
    qpk = float(ph["q_kW"].max())                 # kW
    # 6면에 같은 HRRPUA 를 주면 면별 기여가 면적에 비례하고
    # 6면 합 = HRRPUA x A = 곡선값 이 된다.
    hrrpua = qpk / A
    L.append(f"! 소스: ref_data/{c.hrr.csv}  (t[min], HRR[MW]) 를 선형보간")
    L.append(f"! 6면 합 면적 A = {A:.4f} m2 ({', '.join(vz['faces'])})")
    L.append(f"! pHRR {qpk/1e3:.3f} MW @ {ph['t_peak_min']:.1f} min,  "
             f"THR {ph['thr_GJ']:.3f} GJ,  HRRPUA = pHRR/A = {hrrpua:.2f} kW/m2")
    L.append("! 6면 합 = HRRPUA x A x RAMP_Q(t) = 곡선값. 면적비례는 자동으로 성립한다.")
    s = ["&SURF ID='VENT_PACK', COLOR='ORANGE'",
         f"      HRRPUA={hrrpua:.4f}, RAMP_Q='hrr_pack'"]
    if c.gas.track_hf:
        # HRRPUA 는 연료종에만 걸리고, 비연료종은 MASS_FRACTION 비율로 함께 나간다
        # (User Guide 'Specifying HRRPUA with multiple reactions').
        yhf = float(np.average([z.Y_HF.max() for z in ZONES_1D]))
        s += [f"      SPEC_ID(1)='BATTERY_GAS', MASS_FRACTION(1)={1-yhf:.6f}",
              f"      SPEC_ID(2)='{FDS_ID['HF']}', MASS_FRACTION(2)={yhf:.6f}"]
    else:
        s.append("      SPEC_ID='BATTERY_GAS'")
    tpk = float(max(z.T_vent_K.max() for z in ZONES_1D) - 273.15)
    s.append(f"      TMP_FRONT={tpk:.1f}")
    L.append(",\n".join(s) + " /")
    L += make_ramp("hrr_pack", ph["t_s"], ph["q_kW"] / qpk, c.ramp_eps)

    L.append(HDR("지오메트리 - 타이어 -> 하부 차체(속 빈 쉘) -> 캐빈(프레임+유리)"))
    for n, xb, surf, col in geo["obst"]:
        L.append(f"&OBST ID='{n}', XB={fmt_xb(xb)}, SURF_ID='{surf}', COLOR='{col}' /")
    L.append(f"&OBST ID='PACK', XB={fmt_xb(geo['pack'][1])}, "
             f"SURF_ID='PACK_CASE', COLOR='GRAY 30' /   ! 연소하지 않음")
    L.append("! 하부 차체와 캐빈은 두께 %.3f m 판으로만 이루어져 내부가 비어 있다." % c.frame.shell)
    L.append("! 캐빈 프레임(루프+벨트레일 링+6개 필러)은 서로 맞닿아 하나로 연결된다.")
    L.append("! 좌측 앞·뒷 도어창은 하단을 내려 개구로 두었다 (논문 2.4절 half-opened).")
    L.append("! 팩 6면 전체에 벤트가스 경계조건. 팩은 지면·차체 어디에도 닿지 않게")
    L.append("! 띄워 두었으므로 6면이 모두 기체와 접한다.")
    for vz in geo["vents"]:
        for fn, q in zip(vz["faces"], vz["patches"]):
            L.append(f"&VENT XB={fmt_xb(q)}, SURF_ID='VENT_{vz['name']}' /   ! {fn}")
    L.append("")
    for mb in ("XMIN", "XMAX", "YMIN", "YMAX", "ZMAX"):
        L.append(f"&VENT MB='{mb}', SURF_ID='OPEN' /")
    L.append("&VENT MB='ZMIN', SURF_ID='GROUND' /")

    L.append(HDR("센서 - 논문 Fig. 4(b)~(e) 위치 그대로"))
    zp, zw = Z_PIPE*mm, Z_WIN*mm
    L.append(f"! 천장 수냉파이프 열전대 (z = {zp:.3f} m)")
    for n, x, y in SENSORS_TC_CEIL:
        L.append(f"&DEVC ID='{n}', QUANTITY='THERMOCOUPLE', "
                 f"XYZ={px(x):.3f},{py(y):.3f},{zp:.3f} /")
    L.append(f"! 도어창 중앙높이 열전대 (z = {zw:.3f} m)")
    for n, x, y in SENSORS_TC_WIN:
        L.append(f"&DEVC ID='{n}', QUANTITY='THERMOCOUPLE', "
                 f"XYZ={px(x):.3f},{py(y):.3f},{zw:.3f} /")
    L.append("! Schmidt-Boelter 열유속계 4개. RADIOMETER/GAUGE HEAT FLUX 는 FDS에서")
    L.append("! 고체상 quantity라 벽면이 있어야 한다 (없으면 ERROR(427)). 실제 게이지도")
    L.append("! 수냉 몸체이므로 1셀 블록을 세우고 차량을 향한 면에 붙였다.")
    L.append("!   <id>     = RADIOMETER      (복사만. 논문 'irradiance')")
    L.append("!   <id>_tot = GAUGE HEAT FLUX (복사+대류. Schmidt-Boelter 실측 대응)")
    for b_ in gauge_bodies(c):
        L.append(f"&OBST ID='BODY_{b_['id']}', XB={fmt_xb(b_['body'])}, "
                 f"SURF_ID='HF_GAUGE', COLOR='BLUE' /")
        xyz = ",".join(f"{q:.4f}" for q in b_["xyz"])
        L.append(f"&DEVC ID='{b_['id']}', QUANTITY='RADIOMETER', "
                 f"XYZ={xyz}, IOR={b_['ior']:+d} /")
        L.append(f"&DEVC ID='{b_['id']}_tot', QUANTITY='GAUGE HEAT FLUX', "
                 f"XYZ={xyz}, IOR={b_['ior']:+d} /")

    L.append("")
    L.append("! 반응별 HRR 분리 (논문 Test 1 팩 vs Test 2 차체 기여 대조)")
    dom = fmt_xb((g.xmin, g.xmax, g.ymin, g.ymax, g.zmin, g.zmax))
    for rid_, lab in (("BATT", "HRR_BATT"), ("BODY", "HRR_BODY")):
        L.append(f"&DEVC ID='{lab}', QUANTITY='HRRPUV REAC', REAC_ID='{rid_}', "
                 f"SPATIAL_STATISTIC='VOLUME INTEGRAL', XB={dom} /")
    P = geo["pack"][1]
    L.append("")
    L.append("! 팩 표면 열유속 (1D 되먹임용)")
    for nm, xb, ior in (("PACK_BOT", (P[0],P[1],P[2],P[3],P[4],P[4]), -3),
                        ("PACK_XMIN",(P[0],P[0],P[2],P[3],P[4],P[5]), -1),
                        ("PACK_XMAX",(P[1],P[1],P[2],P[3],P[4],P[5]),  1)):
        L.append(f"&DEVC ID='q_{nm}', QUANTITY='GAUGE HEAT FLUX', XB={fmt_xb(xb)}, "
                 f"IOR={ior}, SPATIAL_STATISTIC='MEAN' /")
        L.append(f"&DEVC ID='T_{nm}', QUANTITY='WALL TEMPERATURE', XB={fmt_xb(xb)}, "
                 f"IOR={ior}, SPATIAL_STATISTIC='MEAN' /")

    L.append("")
    for q in ("GAUGE HEAT FLUX", "WALL TEMPERATURE", "BURNING RATE"):
        L.append(f"&BNDF QUANTITY='{q}' /")
    L.append("")
    for sp in slice_planes(c):
        L.append(f"! {sp['note']}")
        for q, spc in sp["qty"]:
            sid = f", SPEC_ID='{spc}'" if spc else ""
            L.append(f"&SLCF PB{sp['axis'].upper()}={sp['coord']:.4f}, QUANTITY='{q}'{sid} /")
    L.append("&ISOF QUANTITY='HRRPUV', VALUE(1)=100. /")
    L.append("\n&TAIL /")

    p = outdir / f"{c.chid}.fds"
    p.write_text("\n".join(L) + "\n", encoding="utf-8")
    return p


def check_devices(c: Case, fds_path: Path):
    # 생성된 .fds 를 다시 읽어 점 DEVC 가 (a) solid 내부가 아니고
    # (b) 메쉬 경계면 위가 아닌지 확인한다. 둘 다 조용히 틀리기 쉬운 실수다.
    import re
    txt = fds_path.read_text()
    obst = [(m.group(1), [float(q) for q in m.group(2).split(",") if q])
            for m in re.finditer(r"&OBST ID='([^']+)', XB=([-\d\.,]+)", txt)]
    pts = re.findall(r"&DEVC ID='([^']+)',[^\n]*XYZ=([-\d\.,]+)", txt)
    _, planes = mesh_lines(c.grid)
    gids = {b_["id"] for b_ in gauge_bodies(c)}
    gids |= {i + "_tot" for i in gids}
    bad = []
    for nm, cs in pts:
        p_ = [float(q) for q in cs.split(",")[:3]]
        # 게이지 감지점은 게이지 몸체에 '접해' 있어야 정상이라 solid 검사 제외
        if nm not in gids:
            ins = [n for n, bb in obst
                   if bb[0] < p_[0] < bb[1] and bb[2] < p_[1] < bb[3] and bb[4] < p_[2] < bb[5]]
            if ins: bad.append(f"{nm}: solid {ins} 내부")
        for a, key in enumerate("xyz"):
            for pl in planes[key]:
                if abs(p_[a] - pl) < 1e-9:
                    bad.append(f"{nm}: {key} 메쉬 경계면 {pl} 위 (ERROR(427) 위험)")
    if bad:
        raise RuntimeError("DEVC 배치 오류:\n  " + "\n  ".join(bad))
    print(f"  DEVC 배치 검사 통과 ({len(pts)}개 점 센서)")


def write_sbatch(c: Case, outdir: Path) -> Path:
    txt = f"""#!/bin/bash
#SBATCH -J {c.chid}
#SBATCH -p cpu
#SBATCH --nodes=1
#SBATCH --ntasks={c.grid.nranks}
#SBATCH --cpus-per-task=1
#SBATCH --time={c.walltime}
#SBATCH --comment=etc
#SBATCH -o {RESULTS}/{c.chid}/slurm-%j.out
#SBATCH -e {RESULTS}/{c.chid}/slurm-%j.err

source {FDS_ENV}
export OMP_NUM_THREADS=1 I_MPI_FABRICS=shm:ofi I_MPI_PIN=1
RUNDIR={RESULTS}/{c.chid}
mkdir -p "$RUNDIR"
cp {CASEDIR}/{c.chid}/{c.chid}.fds "$RUNDIR"/
cd "$RUNDIR"
echo "host $(hostname)  ranks $SLURM_NTASKS  cells {c.grid.ncell}  start $(date)"
mpiexec -n "$SLURM_NTASKS" "$FDS_BIN" {c.chid}.fds
echo "end $(date)"; tail -n 5 {c.chid}.out
"""
    outdir.mkdir(parents=True, exist_ok=True)
    p = outdir / f"run_{c.chid}.sbatch"; p.write_text(txt); p.chmod(0o755)
    return p

## 8. 케이스 생성

메쉬 분할면은 **센서 좌표를 피해서** 자동으로 고른다. 분할면이 센서와 겹치면
그 DEVC가 어느 메쉬에 속하는지 모호해지고, 특히 열유속계는 몸체와 다른 메쉬에
배정되어 `ERROR(427)` 로 죽는다.

In [ ]:
def safe_cut(axis, target, avoid, g: Grid, min_gap_cells=2):
    lo = (g.xmin, g.ymin, g.zmin)[axis]
    hi = (g.xmax, g.ymax, g.zmax)[axis]
    gap = min_gap_cells * g.dx
    n = round((hi - lo) / g.dx)
    for v in sorted((lo + i*g.dx for i in range(2, n-1)), key=lambda q: abs(q - target)):
        if all(abs(v - a) > gap + 1e-9 for a in avoid):
            return round(v, 6)
    raise RuntimeError(f"axis {axis}: 센서를 피하는 분할면을 못 찾음")


BASE = Case(chid="KANG_T4",
            title="Kang et al. 2023 Applied Energy - Test 4 (BEV_3, 64 kWh) validation",
            t_end=1800.0)

_avoid = {0: set(), 1: set(), 2: {Z_PIPE*mm, Z_WIN*mm}}
for n_, x_, y_ in SENSORS_TC_CEIL + SENSORS_TC_WIN:
    _avoid[0].add(px(x_)); _avoid[1].add(py(y_))
for n_, x_, y_, o_ in SENSORS_HF:
    _avoid[0].add(px(x_)); _avoid[1].add(py(y_))

# 차량 중심(0,0) 근처를 목표로 하되 센서에서 2셀 이상 떨어뜨린다.
_xc = safe_cut(0, 0.0, _avoid[0], BASE.grid)
_yc = safe_cut(1, 0.0, _avoid[1], BASE.grid)
BASE = replace(BASE, grid=replace(BASE.grid, nmesh=(2, 2, 1), cuts=((_xc,), (_yc,), None)))
print(f"메쉬 분할면: x = {_xc:.3f} m, y = {_yc:.3f} m   (센서 좌표에서 2셀 이상 이격)\n")

CASES = {
    "KANG_T4_quick": replace(BASE, chid="KANG_T4_quick", t_end=300.0, walltime="06:00:00",
                             title="Kang et al. Test 4 - smoke test"),
    "KANG_T4": BASE,
}

# HTML 은 커널 없이 남에게 보내거나 향후 인터페이스에 붙일 때 쓴다 (각 4.7 MB).
# 형상만 빠르게 반복 확인할 때는 False 로 두고 5.2절 quick_check() 를 쓰면 된다.
EXPORT_HTML = True

QA = {}
for name, c in CASES.items():
    d = CASEDIR / c.chid
    f = write_fds(c, d)
    s = write_sbatch(c, d)
    QA[name] = geometry_qa(c, verbose=True)
    check_devices(c, f)
    preview_domain(c, save=d / f"{c.chid}_domain.png")
    if EXPORT_HTML:
        preview_plotly(c, save=d / f"{c.chid}_geometry.html")
    plt.close("all")
    (d / "params.json").write_text(json.dumps(
        {"case": asdict(c), "qa": QA[name], "paper": KANG}, ensure_ascii=False,
        indent=2, default=str), encoding="utf-8")
    print(f"  -> {f.relative_to(ROOT)}  ({f.stat().st_size/1024:.1f} kB)")
    html = d / f"{c.chid}_geometry.html"
    if html.exists():
        print(f"  -> {html.relative_to(ROOT)}  ({html.stat().st_size/1024/1024:.1f} MB, "
              f"커널 없이 브라우저에서 열림 / 향후 인터페이스용)")
    print()

## 9. 검증 항목

`results/postprocess.ipynb` 에서 확인. 정답은 1.4절 표.

| # | 항목 | 논문 값 (Test 4) | 비교 방법 |
|---|---|---|---|
| 1 | **pHRR** | 7.25 MW | `_hrr.csv` 최대값 |
| 2 | **pHRR 도달 시각** | 11~17 분 | 최대값 시각 |
| 3 | **THR** | 9.03 GJ | ∫HRR dt |
| 4 | **화재성장계수 θ** | 0.0085~0.020 MW/s² | 성장구간 `Q = θt²` 피팅 |
| 5 | **소실질량 Δm** | 296 kg (17.6 %) | `_mass.csv` |
| 6 | **팩 vs 차체 기여** | 1.30 / 7.53 GJ | `HRR_BATT` / `HRR_BODY` 적분 |
| 7 | **천장 가스온도** | Fig. 9 (피크 > 900 ℃) | `TC_G*` 6개 |
| 8 | **인접차량 입사열유속** | Fig. 10 | `HF_F/R/FD/RD` (+ `_tot`) |

### 주 튜닝 노브

`THICKNESS`(총 THR)는 Table 6 소실질량으로 **이미 고정**되었다. 남은 자유도는
**연소 속도**뿐이고 우선순위는

1. `HEAT_OF_REACTION`(기화열) — 현재 1,800 kJ/kg. pHRR과 성장속도를 직접 지배
2. `BOILING_TEMPERATURE` — 현재 380 ℃. 착화 지연과 화염전파 속도
3. `ABSORPTION_COEFFICIENT` — 표면 흡수 깊이

### 알려진 한계

- 논문 팩 ΔH_c,eff = **45.9 MJ/kg** 은 벤트가스(14.6 MJ/kg)보다 훨씬 크다.
  분리막·전해액 등 **고체 가연분 연소가 포함된 값**이므로, 팩 기여를 벤트가스
  하나로 모사하면 THR 1.30 GJ에 못 미친다(현재 0.40 GJ). 팩에도 `MATL`을 주거나
  벤트 질량을 올려야 한다 — 1D 실데이터 도착 시 재설계할 항목.
- 캐빈 하부(도어·시트 영역)는 여전히 통짜 solid다. 좌측 도어창 개구로 환기는
  되지만 좌석·대시보드 같은 내부 가연물의 개별 배치는 모사하지 않는다.
- 타이어 공기는 논문에서 의도적으로 뺐다(폭발 방지). 모델에는 영향 없음.
- **계산비용:** dx=0.125 m, 245,760 cells, 4 rank 에서 실측 약 **1.9 s wall /
  1 s sim** (`KANG_T4_quick` 실행 결과: 2 h walltime 에 t=232 s 도달).
  1,800 s 해석에 약 **1 일**, 논문의 70분 전체 재현은 2 일 이상. 격자를
  0.15~0.20 m로 완화하거나 rank를 8~16으로 늘려야 한다.